In [1]:
# ============================================================================
# FULL SHORTCUT DIAGNOSTIC + MECHANISM PROBES — with fixes
# ----------------------------------------------------------------------------
# Vanilla     : /kaggle/input/notebooks/sumn2u/vanilla-u2-net/vanilla_v2/vanilla_v2.pth
# Bg-robust   : /kaggle/input/notebooks/sumn2u/background-robust/bg_robust_v3/bg_robust_v3.pth
# Fine-tuned  : auto-discovered under /kaggle/input/notebooks/sumn2u/vanilla-fine-tuned
# Bg-Aug      : auto-discovered under /kaggle/input/notebooks/sumn2u/vanilla-u-net-with-background-augmentation
# Output      : /kaggle/working/eed_test/
#
# Tests: clean, eed, bg_gray, bg_noise, bg_gauss25, bg_blur,
#        bg_only, obj_only, shift25, scale0.6, scale1.5,
#        crop, pad, checker
#
# FIXES vs previous run:
#   [F1] Grad-CAM skips constant CAMs; returns n/a instead of NaN
#   [F2] CDRE now measures task vs SHORTCUT direction (clean-vs-perturbed),
#        not task vs class-id
#   [F3] Summary table handles NaN cleanly
#   [F4] Identical-checkpoint detection: skips duplicate arms from mechanism
# ============================================================================

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONHASHSEED"] = "42"

import gc, json, random, math, importlib.util, traceback, hashlib
from collections import defaultdict
import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

cv2.setNumThreads(0)

# ============================================================
# 0. Config
# ============================================================
CLASSES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]
IMG_EXTS = {".jpg", ".jpeg", ".png"}
MASK_SUFFIX = "_mask.png"

IMG_TRAIN = 320
BATCH = 4
SEED = 42
VAL_FRAC = 0.15
FG_MIN = 0.02
FG_MAX = 0.95
MIN_PER_CLASS = 50
TOL = 10

THRESH_GRID = np.linspace(0.05, 0.95, 19)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ============================================================
# 0b. Paths
# ============================================================
VANILLA_CKPT = "/kaggle/input/notebooks/sumn2u/vanilla-u2-net/vanilla_v2/vanilla_v2.pth"
VANILLA_ARCH = "/kaggle/input/notebooks/sumn2u/vanilla-u2-net/vanilla_v2/u2net_arch.py"

BG_ROBUST_CKPT = "/kaggle/input/notebooks/sumn2u/background-robust/bg_robust_v3/bg_robust_v3.pth"

FINETUNED_DIR = "/kaggle/input/notebooks/sumn2u/vanilla-fine-tuned"
BG_AUG_DIR    = "/kaggle/input/notebooks/sumn2u/vanilla-u-net-with-background-augmentation"

OUT_DIR = "/kaggle/working/eed_test"
os.makedirs(OUT_DIR, exist_ok=True)
MECH_DIR = os.path.join(OUT_DIR, "mechanism")
os.makedirs(MECH_DIR, exist_ok=True)

U2NET_MEAN = [0.485, 0.456, 0.406]
U2NET_STD  = [0.229, 0.224, 0.225]
norm_tf = transforms.Normalize(U2NET_MEAN, U2NET_STD)

print("Vanilla ckpt    :", VANILLA_CKPT, "exists:", os.path.exists(VANILLA_CKPT))
print("Bg-robust ckpt  :", BG_ROBUST_CKPT, "exists:", os.path.exists(BG_ROBUST_CKPT))
print("Fine-tuned dir  :", FINETUNED_DIR, "exists:", os.path.isdir(FINETUNED_DIR))
print("Bg-Aug dir      :", BG_AUG_DIR, "exists:", os.path.isdir(BG_AUG_DIR))
print("Output dir      :", OUT_DIR)

# ============================================================
# 0c. Auto-discover checkpoints + arch
# ============================================================
def discover_ckpt(root):
    if not os.path.isdir(root):
        return None
    candidates = []
    for dirpath, _, files in os.walk(root):
        for f in files:
            if f.lower().endswith((".pth", ".pt", ".ckpt")):
                p = os.path.join(dirpath, f)
                try:
                    sz = os.path.getsize(p)
                except OSError:
                    continue
                candidates.append((sz, p))
    if not candidates:
        return None
    candidates.sort(reverse=True)
    print(f"[discover] candidates under {root}:")
    for sz, p in candidates[:5]:
        print(f"   {sz/1e6:8.2f} MB  {p}")
    preferred = [p for sz, p in candidates if "full_weights" not in os.path.basename(p).lower()]
    if preferred:
        preferred_sorted = sorted(
            [(sz, p) for sz, p in candidates if p in preferred], reverse=True)
        return preferred_sorted[0][1]
    return candidates[0][1]

def discover_arch(root, default):
    if not os.path.isdir(root):
        return default
    for dirpath, _, files in os.walk(root):
        for f in files:
            if f == "u2net_arch.py":
                return os.path.join(dirpath, f)
    return default

FINETUNED_CKPT = discover_ckpt(FINETUNED_DIR)
print("Fine-tuned ckpt :", FINETUNED_CKPT)
FINETUNED_ARCH = discover_arch(FINETUNED_DIR, VANILLA_ARCH)
print("Fine-tuned arch :", FINETUNED_ARCH)

BG_AUG_CKPT = discover_ckpt(BG_AUG_DIR)
print("Bg-Aug ckpt     :", BG_AUG_CKPT)
BG_AUG_ARCH = discover_arch(BG_AUG_DIR, VANILLA_ARCH)
print("Bg-Aug arch     :", BG_AUG_ARCH)

# ============================================================
# 0d. Hash + duplicate detection  [F4]
# ============================================================
def md5_of_file(p):
    if not p or not os.path.isfile(p): return None
    h = hashlib.md5()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

print("\n[hash] checkpoint fingerprints (should be 4 DIFFERENT values):")
HASHES = {}
for label, p in [("vanilla", VANILLA_CKPT), ("bg_robust", BG_ROBUST_CKPT),
                 ("finetuned", FINETUNED_CKPT), ("bg_aug", BG_AUG_CKPT)]:
    h = md5_of_file(p)
    HASHES[label] = h
    print(f"  {label:<10} {h}  {p}")

# detect duplicate-by-hash arms — will EXCLUDE them from mechanism probes
DUPLICATE_ARMS = []
labels = list(HASHES.keys())
for i in range(len(labels)):
    for j in range(i + 1, len(labels)):
        a, b = labels[i], labels[j]
        if HASHES[a] and HASHES[a] == HASHES[b]:
            DUPLICATE_ARMS.append(b)   # keep the first, drop the second
            print(f"  !! {b} is byte-identical to {a} (same MD5) — will be excluded from mechanism probes.")

# ============================================================
# 1. Dataset root discovery
# ============================================================
def find_root(base, required_subdirs):
    if not os.path.isdir(base): return None
    if all(os.path.isdir(os.path.join(base, c)) for c in required_subdirs):
        return base
    for sub in sorted(os.listdir(base)):
        cand = os.path.join(base, sub)
        if os.path.isdir(cand) and all(
            os.path.isdir(os.path.join(cand, c)) for c in required_subdirs):
            return cand
    return None

def locate_trashnet():
    candidates = [
        "/kaggle/input/trashnet-segmentation",
        "/kaggle/input/datasets/sumn2u/trashnet-segmentation",
        "/kaggle/input/trashnet-segmentation/trashnet-segmentation",
    ]
    for c in candidates:
        r = find_root(c, CLASSES)
        if r:
            print(f"[locate] found at: {r}")
            return r
    print("[locate] scanning /kaggle/input ...")
    if os.path.isdir("/kaggle/input"):
        for sub in sorted(os.listdir("/kaggle/input")):
            subp = os.path.join("/kaggle/input", sub)
            if not os.path.isdir(subp): continue
            r = find_root(subp, CLASSES)
            if r:
                print(f"[locate] found at: {r}")
                return r
    raise FileNotFoundError("Could not locate TrashNet class folders under /kaggle/input")

DATA_ROOT = locate_trashnet()

# ============================================================
# 2. Build pairs + val split
# ============================================================
def rgb_mask_to_binary(mask_path, tol=TOL):
    m = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
    if m is None: raise IOError(mask_path)
    if m.ndim == 2: return (m > 127).astype(np.uint8) * 255
    if m.shape[2] == 4: return (m[..., 3] > 127).astype(np.uint8) * 255
    diff = m[..., :3].astype(np.int16).sum(axis=2)
    return (diff > tol).astype(np.uint8) * 255

pairs = []
for cls in CLASSES:
    cp = os.path.join(DATA_ROOT, cls)
    if not os.path.isdir(cp): continue
    for f in sorted(os.listdir(cp)):
        if not f.lower().endswith(MASK_SUFFIX): continue
        stem = f[:-len(MASK_SUFFIX)]
        img = None
        for ext in IMG_EXTS:
            cand = os.path.join(cp, stem + ext)
            if os.path.isfile(cand): img = cand; break
        if img is None: continue
        try:
            b = rgb_mask_to_binary(os.path.join(cp, f))
        except Exception:
            continue
        fg = (b > 127).mean()
        if not (FG_MIN < fg < FG_MAX): continue
        pairs.append((img, b.copy(), cls))

counts = pd.Series([p[2] for p in pairs]).value_counts()
CLASSES_TRAINING = [c for c in CLASSES if counts.get(c, 0) >= MIN_PER_CLASS]
pairs = [p for p in pairs if p[2] in CLASSES_TRAINING]

random.seed(SEED); random.shuffle(pairs)
by_class = defaultdict(list)
for p in pairs: by_class[p[2]].append(p)
train_pairs, val_pairs = [], []
for cls, items in by_class.items():
    n_val = max(1, int(round(VAL_FRAC * len(items))))
    val_pairs.extend(items[:n_val])
    train_pairs.extend(items[n_val:])
random.shuffle(train_pairs); random.shuffle(val_pairs)
assert not (set(p[0] for p in train_pairs) & set(p[0] for p in val_pairs))
print(f"Train: {len(train_pairs)}   Val: {len(val_pairs)}")
print(f"Classes used: {CLASSES_TRAINING}")

# ============================================================
# 3. Transformation library
# ============================================================
def _crop_around_object(img, msk, margin=0.25, out_size=320):
    ys, xs = np.where(msk > 127)
    if len(ys) == 0:
        return (cv2.resize(img, (out_size, out_size)),
                cv2.resize(msk, (out_size, out_size), interpolation=cv2.INTER_NEAREST))
    H, W = img.shape[:2]
    y0, y1 = ys.min(), ys.max(); x0, x1 = xs.min(), xs.max()
    my = int(margin * (y1 - y0 + 1)); mx = int(margin * (x1 - x0 + 1))
    y0 = max(0, y0 - my); y1 = min(H - 1, y1 + my)
    x0 = max(0, x0 - mx); x1 = min(W - 1, x1 + mx)
    ci = img[y0:y1+1, x0:x1+1]; cm = msk[y0:y1+1, x0:x1+1]
    h, w = ci.shape[:2]; s = out_size / max(h, w)
    nh, nw = int(h*s), int(w*s)
    ci = cv2.resize(ci, (nw, nh))
    cm = cv2.resize(cm, (nw, nh), interpolation=cv2.INTER_NEAREST)
    canvas_i = np.full((out_size, out_size, 3), 255, np.uint8)
    canvas_m = np.zeros((out_size, out_size), np.uint8)
    oy, ox = (out_size - nh)//2, (out_size - nw)//2
    canvas_i[oy:oy+nh, ox:ox+nw] = ci
    canvas_m[oy:oy+nh, ox:ox+nw] = cm
    return canvas_i, canvas_m

def _pad_to_size(img, msk, out_size=320, pad_frac=0.3):
    h, w = img.shape[:2]
    ph, pw = int(pad_frac*h), int(pad_frac*w)
    ip = np.full((h+2*ph, w+2*pw, 3), 255, np.uint8)
    mp = np.zeros((h+2*ph, w+2*pw), np.uint8)
    ip[ph:ph+h, pw:pw+w] = img; mp[ph:ph+h, pw:pw+w] = msk
    return (cv2.resize(ip, (out_size, out_size)),
            cv2.resize(mp, (out_size, out_size), interpolation=cv2.INTER_NEAREST))

def make_eed(img):
    g = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    return cv2.cvtColor(cv2.Canny(g, 50, 150), cv2.COLOR_GRAY2RGB)

def make_bg_gray(img, msk):
    out = img.copy(); out[msk <= 127] = 128; return out

def make_bg_noise(img, msk, rng):
    out = img.copy()
    noise = rng.randint(0, 256, size=img.shape, dtype=np.uint8)
    out[msk <= 127] = noise[msk <= 127]; return out

def make_bg_gauss(img, msk, sigma=25, rng=None):
    rng = rng or np.random.RandomState(0)
    out = img.copy()
    noise = rng.normal(0, sigma, img.shape).astype(np.int16)
    bg = (msk <= 127)
    out[bg] = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)[bg]
    return out

def make_bg_blur(img, msk):
    out = img.copy()
    blurred = cv2.GaussianBlur(img, (31, 31), 0)
    out[msk <= 127] = blurred[msk <= 127]; return out

def make_bg_checker(img, msk, tile=16):
    H, W = img.shape[:2]
    pat = ((np.indices((H, W)).sum(0) // tile) % 2 * 255).astype(np.uint8)
    pat3 = np.stack([pat]*3, -1)
    out = img.copy(); out[msk <= 127] = pat3[msk <= 127]; return out

def make_bg_only(img, msk, fill=128):
    out = img.copy(); out[msk > 127] = fill; return out

def make_obj_only(img, msk, fill=128):
    out = np.full_like(img, fill); out[msk > 127] = img[msk > 127]; return out

def shift_img_mask(img, msk, dx, dy):
    M = np.float32([[1, 0, dx], [0, 1, dy]])
    h, w = img.shape[:2]
    return (cv2.warpAffine(img, M, (w, h), borderValue=(255,255,255)),
            cv2.warpAffine(msk, M, (w, h), borderValue=0))

def scale_object(img, msk, scale):
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w/2, h/2), 0, scale)
    return (cv2.warpAffine(img, M, (w, h), borderValue=(255,255,255)),
            cv2.warpAffine(msk, M, (w, h), borderValue=0))

def voronoi_shuffle(img, n_cells=80, seed=0):
    rng = np.random.RandomState(seed)
    H, W = img.shape[:2]
    seeds = np.stack([rng.randint(0, H, n_cells), rng.randint(0, W, n_cells)], 1)
    yy, xx = np.mgrid[0:H, 0:W]
    d = (yy[..., None] - seeds[:, 0])**2 + (xx[..., None] - seeds[:, 1])**2
    lab = d.argmin(-1)
    perm = rng.permutation(n_cells)
    out = np.zeros_like(img)
    for c in range(n_cells):
        src = (lab == perm[c]); dst = (lab == c)
        if src.sum() == 0 or dst.sum() == 0: continue
        ys_s, xs_s = np.where(src); ys_d, xs_d = np.where(dst)
        k = min(len(ys_s), len(ys_d))
        out[ys_d[:k], xs_d[:k]] = img[ys_s[:k], xs_s[:k]]
    return out

# ============================================================
# 4. Dataset
# ============================================================
class DiagDS(Dataset):
    def __init__(self, pairs, size=IMG_TRAIN, variant="clean",
                 sigma=25, dx=0, dy=0, scale=1.0, tile=16):
        self.pairs = pairs; self.size = size; self.variant = variant
        self.sigma = sigma; self.dx = dx; self.dy = dy
        self.scale = scale; self.tile = tile
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        ip, bin_mask, _ = self.pairs[i]
        img = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
        msk = cv2.resize(bin_mask, (img.shape[1], img.shape[0]),
                         interpolation=cv2.INTER_NEAREST)
        if self.variant == "crop":
            img, msk = _crop_around_object(img, msk, 0.25, self.size)
        elif self.variant == "pad":
            img, msk = _pad_to_size(img, msk, self.size, 0.3)
        else:
            img = cv2.resize(img, (self.size, self.size))
            msk = cv2.resize(msk, (self.size, self.size),
                             interpolation=cv2.INTER_NEAREST)
        if self.variant == "eed":
            img = make_eed(img)
        elif self.variant == "bg_gray":
            img = make_bg_gray(img, msk)
        elif self.variant == "bg_noise":
            img = make_bg_noise(img, msk, np.random.RandomState(SEED + i))
        elif self.variant == "bg_gauss":
            img = make_bg_gauss(img, msk, self.sigma, np.random.RandomState(SEED + i))
        elif self.variant == "bg_blur":
            img = make_bg_blur(img, msk)
        elif self.variant == "bg_checker":
            img = make_bg_checker(img, msk, tile=self.tile)
        elif self.variant == "bg_only":
            img = make_bg_only(img, msk)
        elif self.variant == "obj_only":
            img = make_obj_only(img, msk)
        elif self.variant == "shift":
            img, msk = shift_img_mask(img, msk, self.dx, self.dy)
            img = cv2.resize(img, (self.size, self.size))
            msk = cv2.resize(msk, (self.size, self.size),
                             interpolation=cv2.INTER_NEAREST)
        elif self.variant == "scale":
            img, msk = scale_object(img, msk, self.scale)
        x = norm_tf(transforms.ToTensor()(Image.fromarray(img)))
        y = (torch.from_numpy(msk).float().unsqueeze(0) > 127).float()
        return x, y

def make_loader(pairs, variant, **kw):
    ds = DiagDS(pairs, variant=variant, **kw)
    g = torch.Generator(); g.manual_seed(SEED)
    return DataLoader(ds, batch_size=BATCH, shuffle=False,
                      num_workers=2, pin_memory=True, generator=g)

# ============================================================
# 5. Architecture (U2NET loaded from arch file; BG-robust inlined)
# ============================================================
def load_arch(arch_path):
    spec = importlib.util.spec_from_file_location("u2net_arch", arch_path)
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
    return mod.U2NET

def _upsample_like(x, ref):
    size = tuple(ref.shape[-2:]) if torch.is_tensor(ref) else tuple(ref)
    return nn.Upsample(size=size, mode='bilinear', align_corners=False)(x)

def _size_map(x, height):
    size = list(x.shape[-2:]); sizes = {}
    for h in range(1, height):
        sizes[h] = size
        size = [math.ceil(w / 2) for w in size]
    return sizes

class REBNCONV(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, dilate=1):
        super().__init__()
        self.conv_s1 = nn.Conv2d(in_ch, out_ch, 3, padding=dilate, dilation=dilate)
        self.bn_s1 = nn.BatchNorm2d(out_ch); self.relu_s1 = nn.ReLU(inplace=True)
    def forward(self, x): return self.relu_s1(self.bn_s1(self.conv_s1(x)))

class MultiScaleExpert(nn.Module):
    def __init__(self, in_ch, out_ch, dilations=(1, 2, 4)):
        super().__init__()
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=d, dilation=d, bias=False),
                          nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
            for d in dilations])
        self.gate = nn.Conv2d(in_ch, len(dilations), 1)
    def forward(self, x):
        outs = [e(x) for e in self.experts]
        stacked = torch.stack(outs, dim=1)
        gate_w = F.softmax(self.gate(x), dim=1).unsqueeze(2)
        return (stacked * gate_w).sum(dim=1)

class RSU(nn.Module):
    def __init__(self, name, height, in_ch, mid_ch, out_ch, use_moe=False):
        super().__init__()
        self.name = name; self.height = height; self.use_moe = use_moe
        self._make_layers(in_ch, mid_ch, out_ch, use_moe)
    def forward(self, x):
        sizes = _size_map(x, self.height)
        x0 = self.rebnconvin(x)
        def unet(x, height=1):
            if height < self.height:
                x1 = getattr(self, f'rebnconv{height}')(x)
                if height < self.height - 1:
                    x2 = unet(getattr(self, 'downsample')(x1), height + 1)
                else:
                    x2 = self.moe(x1) if self.use_moe else x1
                x = getattr(self, f'rebnconv{height}d')(torch.cat((x2, x1), 1))
                return _upsample_like(x, sizes[height - 1]) if height > 1 else x
            return getattr(self, f'rebnconv{height}')(x)
        return x0 + unet(x0)
    def _make_layers(self, in_ch, mid_ch, out_ch, use_moe):
        self.add_module('rebnconvin', REBNCONV(in_ch, out_ch))
        self.add_module('downsample', nn.MaxPool2d(2, stride=2, ceil_mode=True))
        self.add_module('rebnconv1', REBNCONV(out_ch, mid_ch))
        self.add_module('rebnconv1d', REBNCONV(mid_ch*2, out_ch))
        for i in range(2, self.height):
            self.add_module(f'rebnconv{i}', REBNCONV(mid_ch, mid_ch))
            self.add_module(f'rebnconv{i}d', REBNCONV(mid_ch*2, mid_ch))
        self.add_module(f'rebnconv{self.height}', REBNCONV(mid_ch, mid_ch))
        if use_moe:
            self.moe = MultiScaleExpert(mid_ch, mid_ch, dilations=(1, 2, 4))

class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, 1, bias=True), nn.BatchNorm2d(F_int))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, 1, bias=True), nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1, bias=True),
                                  nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        psi = self.psi(self.relu(self.W_g(g) + self.W_x(x)))
        return x * psi

class FactorizedHead(nn.Module):
    def __init__(self, in_ch=64, mid_ch=64):
        super().__init__()
        self.object_branch = nn.Sequential(
            nn.Conv2d(in_ch, mid_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(mid_ch), nn.ReLU(inplace=True),
            nn.Conv2d(mid_ch, mid_ch, 3, padding=1, bias=False))
        self.context_branch = nn.Sequential(
            nn.Conv2d(in_ch, mid_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(mid_ch), nn.ReLU(inplace=True),
            nn.Conv2d(mid_ch, mid_ch, 3, padding=1, bias=False))
    def forward(self, feat): return self.object_branch(feat), self.context_branch(feat)

class BackgroundRobustU2Net(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, n_classes=6):
        super().__init__()
        self.stage1 = RSU('stage1', 7, in_ch, 32, 64); self.pool12 = nn.MaxPool2d(2, 2, ceil_mode=True)
        self.stage2 = RSU('stage2', 6, 64, 32, 128);   self.pool23 = nn.MaxPool2d(2, 2, ceil_mode=True)
        self.stage3 = RSU('stage3', 5, 128, 64, 256);  self.pool34 = nn.MaxPool2d(2, 2, ceil_mode=True)
        self.stage4 = RSU('stage4', 4, 256, 128, 512); self.pool45 = nn.MaxPool2d(2, 2, ceil_mode=True)
        self.stage5 = RSU('stage5', 4, 512, 256, 512, use_moe=True)
        self.pool56 = nn.MaxPool2d(2, 2, ceil_mode=True)
        self.stage6 = RSU('stage6', 4, 512, 256, 512, use_moe=True)
        self.stage5d = RSU('stage5d', 4, 1024, 256, 512)
        self.stage4d = RSU('stage4d', 4, 1024, 128, 256)
        self.stage3d = RSU('stage3d', 5, 512, 64, 128)
        self.stage2d = RSU('stage2d', 6, 256, 32, 64)
        self.stage1d = RSU('stage1d', 7, 128, 16, 64)
        self.att5 = AttentionGate(512, 512, 256)
        self.att4 = AttentionGate(512, 512, 256)
        self.att3 = AttentionGate(256, 256, 128)
        self.factor_head = FactorizedHead(64, 64)
        self.fg_frac_head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                           nn.Linear(512, 64), nn.ReLU(inplace=True),
                                           nn.Linear(64, 1), nn.Sigmoid())
        self.ctx_classifier = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                             nn.Linear(64, 64), nn.ReLU(inplace=True),
                                             nn.Linear(64, n_classes))
        self.side1 = nn.Conv2d(64, out_ch, 3, padding=1)
        self.side2 = nn.Conv2d(64, out_ch, 3, padding=1)
        self.side3 = nn.Conv2d(128, out_ch, 3, padding=1)
        self.side4 = nn.Conv2d(256, out_ch, 3, padding=1)
        self.side5 = nn.Conv2d(512, out_ch, 3, padding=1)
        self.side6 = nn.Conv2d(512, out_ch, 3, padding=1)
        self.outconv = nn.Conv2d(6 * out_ch, out_ch, 1)
    def forward(self, x, return_aux=False):
        hx1 = self.stage1(x); hx = self.pool12(hx1)
        hx2 = self.stage2(hx); hx = self.pool23(hx2)
        hx3 = self.stage3(hx); hx = self.pool34(hx3)
        hx4 = self.stage4(hx); hx = self.pool45(hx4)
        hx5 = self.stage5(hx); hx = self.pool56(hx5)
        hx6 = self.stage6(hx)
        hx6up = _upsample_like(hx6, hx5)
        hx5d = self.stage5d(torch.cat((hx6up, self.att5(g=hx6up, x=hx5)), 1))
        hx5dup = _upsample_like(hx5d, hx4)
        hx4d = self.stage4d(torch.cat((hx5dup, self.att4(g=hx5dup, x=hx4)), 1))
        hx4dup = _upsample_like(hx4d, hx3)
        hx3d = self.stage3d(torch.cat((hx4dup, self.att3(g=hx4dup, x=hx3)), 1))
        hx3dup = _upsample_like(hx3d, hx2)
        hx2d = self.stage2d(torch.cat((hx3dup, hx2), 1))
        hx2dup = _upsample_like(hx2d, hx1)
        hx1d = self.stage1d(torch.cat((hx2dup, hx1), 1))
        obj_feat, ctx_feat = self.factor_head(hx1d)
        d1 = self.side1(obj_feat)
        d2 = _upsample_like(self.side2(hx2d), d1)
        d3 = _upsample_like(self.side3(hx3d), d1)
        d4 = _upsample_like(self.side4(hx4d), d1)
        d5 = _upsample_like(self.side5(hx5d), d1)
        d6 = _upsample_like(self.side6(hx6), d1)
        d0 = self.outconv(torch.cat((d1, d2, d3, d4, d5, d6), 1))
        if return_aux:
            return (d0, d1, d2, d3, d4, d5, d6,
                    self.fg_frac_head(hx6),
                    self.ctx_classifier(ctx_feat),
                    obj_feat.mean(dim=(2, 3)))
        return d0, d1, d2, d3, d4, d5, d6

# ============================================================
# 6. Model loaders
# ============================================================
def load_vanilla_arch_model(ckpt_path, arch_path, label):
    if not ckpt_path or not os.path.isfile(ckpt_path):
        print(f"[{label}] checkpoint missing: {ckpt_path}"); return None
    try:
        BaseU2NET = load_arch(arch_path)
        model = BaseU2NET().to(DEVICE)
        sd = torch.load(ckpt_path, map_location="cpu")
        if isinstance(sd, dict) and "state_dict" in sd: sd = sd["state_dict"]
        missing, unexpected = model.load_state_dict(sd, strict=False)
        if missing: print(f"[{label}] missing: {len(missing)}")
        if unexpected: print(f"[{label}] unexpected: {len(unexpected)}")
        if not missing and not unexpected:
            print(f"[{label}] checkpoint loaded cleanly.")
        del sd; gc.collect(); torch.cuda.empty_cache()
        model.eval(); return model
    except Exception as e:
        print(f"[{label}] failed: {e}"); traceback.print_exc(); return None

def load_bg_robust(ckpt_path, n_classes=6):
    if not ckpt_path or not os.path.isfile(ckpt_path):
        print(f"[bg_robust] checkpoint missing: {ckpt_path}"); return None
    model = BackgroundRobustU2Net(n_classes=n_classes).to(DEVICE)
    sd = torch.load(ckpt_path, map_location="cpu")
    if isinstance(sd, dict) and "state_dict" in sd: sd = sd["state_dict"]
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if missing: print(f"[bg_robust] missing: {len(missing)}")
    if unexpected: print(f"[bg_robust] unexpected: {len(unexpected)}")
    if not missing and not unexpected:
        print("[bg_robust] checkpoint loaded cleanly.")
    del sd; gc.collect(); torch.cuda.empty_cache()
    model.eval(); return model

# ============================================================
# 7. Eval
# ============================================================
@torch.no_grad()
def eval_iou(model, loader, thr=0.5):
    model.eval(); ious = []
    for x, y in loader:
        x = x.to(DEVICE)
        d0, *_ = model(x)
        p = (torch.sigmoid(d0) > thr).float().cpu()
        y = y.cpu()
        for k in range(p.shape[0]):
            pk, yk = p[k, 0], y[k, 0]
            inter = (pk * yk).sum().item()
            union = ((pk + yk) > 0).sum().item()
            ious.append(inter / union if union > 0 else 0.0)
    return float(np.mean(ious))

def best_threshold_on_clean(model, pairs, thresholds=THRESH_GRID):
    loader = make_loader(pairs, "clean")
    ious = [eval_iou(model, loader, thr=float(t)) for t in thresholds]
    idx = int(np.argmax(ious))
    return float(thresholds[idx]), float(ious[idx]), [float(v) for v in ious]

@torch.no_grad()
def dump_predictions(model, pairs, variant, tag, n=8, thr=0.5, extra_kw=None):
    extra_kw = extra_kw or {}
    model.eval()
    loader = make_loader(pairs[:n], variant, **extra_kw)
    fig, axes = plt.subplots(n, 3, figsize=(10, 3.2*n))
    if n == 1: axes = np.array([axes])
    for row, (x, y) in enumerate(loader):
        if row >= n: break
        x = x.to(DEVICE)
        d0, *_ = model(x)
        p = (torch.sigmoid(d0) > thr).float().cpu()[0, 0]
        img = x[0].cpu().permute(1, 2, 0).numpy()
        img = (img * np.array(U2NET_STD) + np.array(U2NET_MEAN)).clip(0, 1)
        axes[row, 0].imshow(img);                  axes[row, 0].set_title(f"{variant} input"); axes[row, 0].axis("off")
        axes[row, 1].imshow(y[0, 0], cmap="gray"); axes[row, 1].set_title("GT");                axes[row, 1].axis("off")
        axes[row, 2].imshow(p,      cmap="gray"); axes[row, 2].set_title("pred");              axes[row, 2].axis("off")
    plt.tight_layout()
    out = os.path.join(OUT_DIR, f"preds_{variant}_{tag}.png")
    plt.savefig(out, dpi=120); plt.close()
    print(f"saved -> {out}")

# ============================================================
# 8. Full diagnostic suite
# ============================================================
TESTS = [
    ("clean",     "clean",     {},                "clean"),
    ("eed",       "eed",       {},                "texture"),
    ("bg_gray",   "bg_gray",   {},                "background"),
    ("bg_noise",  "bg_noise",  {},                "background"),
    ("bg_gauss25","bg_gauss",  {"sigma": 25},     "background"),
    ("bg_blur",   "bg_blur",   {},                "background"),
    ("bg_checker","bg_checker",{"tile": 16},      "background"),
    ("bg_only",   "bg_only",   {},                "background"),
    ("obj_only",  "obj_only",  {},                "object"),
    ("shift25",   "shift",     {"dx": 25, "dy": 0},"geometry"),
    ("scale0.6",  "scale",     {"scale": 0.6},    "geometry"),
    ("scale1.5",  "scale",     {"scale": 1.5},    "geometry"),
    ("crop",      "crop",      {},                "geometry"),
    ("pad",       "pad",       {},                "geometry"),
]

def run_full_diagnostics(name, model):
    if model is None:
        print(f"\n[{name}] model is None — skipping."); return None
    print(f"\n{'='*70}\nFULL DIAGNOSTIC — {name.upper()}\n{'='*70}")
    thr, clean_iou, sweep = best_threshold_on_clean(model, val_pairs)
    print(f"[{name}] best threshold: {thr:.2f}  clean IoU={clean_iou:.4f}")
    results = {
        "best_threshold_clean": thr, "clean_iou": clean_iou,
        "threshold_sweep_clean": {"thresholds": [float(t) for t in THRESH_GRID], "ious": sweep},
        "tests": {},
    }
    for tname, variant, kw, _dtype in TESTS:
        iou = eval_iou(model, make_loader(val_pairs, variant, **kw), thr=thr)
        results["tests"][tname] = iou
        print(f"[{name}] {tname:<12} : {iou:.4f}")
    dump_predictions(model, val_pairs, "eed",     name, thr=thr)
    dump_predictions(model, val_pairs, "bg_only", name, thr=thr)
    return results

all_results = {}
mech_models = {}

# --- Vanilla ---
vanilla = load_vanilla_arch_model(VANILLA_CKPT, VANILLA_ARCH, "vanilla")
all_results["vanilla"] = run_full_diagnostics("vanilla", vanilla)
if vanilla is not None: mech_models["vanilla"] = vanilla

# --- Bg-Robust ---
bg = load_bg_robust(BG_ROBUST_CKPT, n_classes=len(CLASSES_TRAINING))
all_results["bg_robust"] = run_full_diagnostics("bg_robust", bg)
if bg is not None: mech_models["bg_robust"] = bg

# --- Fine-tuned ---
finetuned = load_vanilla_arch_model(FINETUNED_CKPT, FINETUNED_ARCH, "finetuned") \
            if FINETUNED_CKPT else None
all_results["finetuned"] = run_full_diagnostics("finetuned", finetuned)
# [F4] only add to mech_models if not a duplicate
if finetuned is not None and "finetuned" not in DUPLICATE_ARMS:
    mech_models["finetuned"] = finetuned

# --- Bg-Aug ---
bg_aug = load_vanilla_arch_model(BG_AUG_CKPT, BG_AUG_ARCH, "bg_aug") \
         if BG_AUG_CKPT else None
all_results["bg_aug"] = run_full_diagnostics("bg_aug", bg_aug)
if bg_aug is not None and "bg_aug" not in DUPLICATE_ARMS:
    mech_models["bg_aug"] = bg_aug

# ============================================================
# 9. Plots
# ============================================================
model_names = [n for n, r in all_results.items() if r is not None]
test_names = [t[0] for t in TESTS]
matrix = np.zeros((len(model_names), len(test_names)))
for i, mn in enumerate(model_names):
    for j, tn in enumerate(test_names):
        matrix[i, j] = all_results[mn]["tests"][tn]

# 9a. Heatmap
fig, ax = plt.subplots(figsize=(1.4*len(test_names)+3, 0.6*len(model_names)+3))
im = ax.imshow(matrix, aspect="auto", cmap="RdYlGn", vmin=0.3, vmax=1.0)
ax.set_xticks(range(len(test_names))); ax.set_xticklabels(test_names, rotation=45, ha="right")
ax.set_yticks(range(len(model_names))); ax.set_yticklabels(model_names)
for i in range(len(model_names)):
    for j in range(len(test_names)):
        v = matrix[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                color="white" if v < 0.55 else "black", fontsize=9)
plt.colorbar(im, ax=ax, label="IoU")
ax.set_title("Shortcut matrix: model (rows) × test (cols)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "shortcut_matrix.png"), dpi=130); plt.close()
print(f"saved -> {os.path.join(OUT_DIR, 'shortcut_matrix.png')}")

# 9b. Drops vs clean
fig, ax = plt.subplots(figsize=(1.4*len(test_names)+3, 0.6*len(model_names)+3))
width = 0.8 / max(len(model_names), 1)
x = np.arange(len(test_names))
for i, mn in enumerate(model_names):
    clean = all_results[mn]["clean_iou"]
    drops = [all_results[mn]["tests"][tn] - clean for tn in test_names]
    ax.bar(x + i*width - (len(model_names)-1)*width/2, drops, width=width, label=mn)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x); ax.set_xticklabels(test_names, rotation=45, ha="right")
ax.set_ylabel("IoU − clean IoU"); ax.set_title("Performance drop vs clean")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "shortcut_drops.png"), dpi=130); plt.close()
print(f"saved -> {os.path.join(OUT_DIR, 'shortcut_drops.png')}")

# 9c. Threshold sweeps
fig, ax = plt.subplots(figsize=(7, 4.5))
for mn in model_names:
    sweep = all_results[mn]["threshold_sweep_clean"]
    ax.plot(sweep["thresholds"], sweep["ious"], marker="o", label=mn)
for mn, t in {mn: all_results[mn]["best_threshold_clean"] for mn in model_names}.items():
    ax.axvline(t, linestyle="--", alpha=0.4)
ax.set_xlabel("threshold"); ax.set_ylabel("clean val IoU")
ax.set_title("Threshold sweeps on clean val"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "threshold_sweep.png"), dpi=130); plt.close()
print(f"saved -> {os.path.join(OUT_DIR, 'threshold_sweep.png')}")

# 9d. EED probe
fig, ax = plt.subplots(figsize=(7, 4.5))
eed_vals = [all_results[mn]["tests"]["eed"] for mn in model_names]
clean_vals = [all_results[mn]["clean_iou"] for mn in model_names]
x = np.arange(len(model_names))
ax.bar(x - 0.2, clean_vals, 0.4, label="clean")
ax.bar(x + 0.2, eed_vals,   0.4, label="eed")
for i, (c, e) in enumerate(zip(clean_vals, eed_vals)):
    ax.text(i-0.2, c+0.01, f"{c:.2f}", ha="center", fontsize=9)
    ax.text(i+0.2, e+0.01, f"{e:.2f}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(model_names)
ax.set_ylabel("IoU"); ax.set_ylim(0, 1.05)
ax.set_title("Texture shortcut probe: clean vs EED")
ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "eed_probe.png"), dpi=130); plt.close()
print(f"saved -> {os.path.join(OUT_DIR, 'eed_probe.png')}")

# ============================================================
# 10. Text summary + JSON
# ============================================================
print("\n" + "=" * 70)
print("FULL DIAGNOSTIC SUMMARY")
print("=" * 70)
header = f"{'test':<12}" + "".join(f"{mn:>14}" for mn in model_names)
print(header); print("-"*len(header))
for tn in test_names:
    row = f"{tn:<12}"
    for mn in model_names: row += f"{all_results[mn]['tests'][tn]:>14.4f}"
    print(row)

print("\nDeltas vs vanilla:")
if "vanilla" in all_results and all_results["vanilla"] is not None:
    v = all_results["vanilla"]["tests"]
    for mn in model_names:
        if mn == "vanilla": continue
        print(f"  [{mn}]")
        for tn in test_names:
            print(f"    {tn:<12} {all_results[mn]['tests'][tn] - v[tn]:+.4f}")

if all_results.get("bg_aug") and all_results.get("bg_robust"):
    a = all_results["bg_aug"]["tests"]; b = all_results["bg_robust"]["tests"]
    print("\nBg-Robust − Bg-Aug (architecture effect over data aug):")
    for tn in test_names: print(f"    {tn:<12} {b[tn] - a[tn]:+.4f}")

if all_results.get("bg_aug") and all_results.get("vanilla"):
    a = all_results["bg_aug"]["tests"]; v = all_results["vanilla"]["tests"]
    print("\nBg-Aug − Vanilla (pure data augmentation effect):")
    for tn in test_names: print(f"    {tn:<12} {a[tn] - v[tn]:+.4f}")

with open(os.path.join(OUT_DIR, "shortcut_summary.json"), "w") as f:
    json.dump(all_results, f, indent=2, default=float)
print(f"\nSaved -> {os.path.join(OUT_DIR, 'shortcut_summary.json')}")

# ============================================================
# 11. MECHANISM PROBES (why + what)  [F1][F2][F3]
# ============================================================
print(f"\n{'='*70}\nMECHANISM PROBES (why + what)\n{'='*70}")
print(f"[mechanism] models to probe: {list(mech_models.keys())}")
if DUPLICATE_ARMS:
    print(f"[mechanism] excluded (duplicate hash): {DUPLICATE_ARMS}")

# --- 11.1 Cue-conflict (Baker design) ---
def _texture_patches(pairs, cls, n=200, patch=32):
    pool = []; rng = np.random.RandomState(SEED)
    for ip, msk, c in pairs:
        if c != cls: continue
        img = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
        m = cv2.resize(msk, (img.shape[1], img.shape[0]),
                       interpolation=cv2.INTER_NEAREST)
        ys, xs = np.where(m > 127)
        if len(ys) < patch: continue
        for _ in range(3):
            i = rng.randint(0, len(ys)); y, x = ys[i], xs[i]
            y0, y1 = max(0, y-patch//2), min(img.shape[0], y+patch//2)
            x0, x1 = max(0, x-patch//2), min(img.shape[1], x+patch//2)
            if y1-y0 < patch or x1-x0 < patch: continue
            pool.append(img[y0:y0+patch, x0:x0+patch])
            if len(pool) >= n: return pool
    return pool

def make_texture_swapped(img, msk, texture_pool, patch=32):
    out = img.copy(); H, W = img.shape[:2]
    ys, xs = np.where(msk > 127)
    if len(ys) == 0 or not texture_pool: return out
    y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
    rng = np.random.RandomState(SEED)
    for yy in range(y0, y1+1, patch):
        for xx in range(x0, x1+1, patch):
            p = texture_pool[rng.randint(len(texture_pool))]
            ph, pw = p.shape[:2]
            y2, x2 = min(yy+ph, H), min(xx+pw, W)
            if y2 <= yy or x2 <= xx: continue
            sub = p[:y2-yy, :x2-xx]
            mask_sub = msk[yy:y2, xx:x2] > 127
            out[yy:y2, xx:x2][mask_sub] = sub[mask_sub]
    return out

@torch.no_grad()
def cue_conflict_test(model, pairs, n=64, patch=32):
    pools = {c: _texture_patches(pairs, c) for c in CLASSES_TRAINING}
    ious = []
    for ip, bin_mask, cls in pairs[:n]:
        img = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_TRAIN, IMG_TRAIN))
        msk = cv2.resize(bin_mask, (IMG_TRAIN, IMG_TRAIN),
                         interpolation=cv2.INTER_NEAREST)
        others = [c for c in CLASSES_TRAINING if c != cls]
        if not others: continue
        src = others[np.random.RandomState(SEED + abs(hash(ip)) % 10000).randint(len(others))]
        sw = make_texture_swapped(img, msk, pools.get(src, []), patch)
        x = norm_tf(transforms.ToTensor()(Image.fromarray(sw))).unsqueeze(0).to(DEVICE)
        d0, *_ = model(x)
        p = (torch.sigmoid(d0) > 0.5).float().cpu()[0, 0].numpy()
        gt = (msk > 127).astype(np.float32)
        inter = (p * gt).sum(); union = ((p + gt) > 0).sum()
        ious.append(float(inter/union) if union > 0 else 0.0)
    return float(np.mean(ious)) if ious else float("nan")

# --- 11.2 Voronoi OOD ---
@torch.no_grad()
def voronoi_ood_iou(model, pairs, n=64):
    ious = []
    for ip, bin_mask, _ in pairs[:n]:
        img = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_TRAIN, IMG_TRAIN))
        msk = cv2.resize(bin_mask, (IMG_TRAIN, IMG_TRAIN),
                         interpolation=cv2.INTER_NEAREST)
        v = voronoi_shuffle(img, n_cells=80, seed=SEED + abs(hash(ip)) % 10000)
        x = norm_tf(transforms.ToTensor()(Image.fromarray(v))).unsqueeze(0).to(DEVICE)
        d0, *_ = model(x)
        p = (torch.sigmoid(d0) > 0.5).float().cpu()[0, 0].numpy()
        gt = (msk > 127).astype(np.float32)
        inter = (p * gt).sum(); union = ((p + gt) > 0).sum()
        ious.append(float(inter/union) if union > 0 else 0.0)
    return float(np.mean(ious)) if ious else float("nan")

# --- 11.3 Shape / texture dimensionality ---
def _layer_feats(model, imgs, layer_name):
    store = {}
    layer = dict(model.named_modules())[layer_name]
    h = layer.register_forward_hook(lambda m, i, o: store.__setitem__("f", o.detach()))
    feats = []
    with torch.no_grad():
        for im in imgs:
            x = norm_tf(transforms.ToTensor()(Image.fromarray(im))).unsqueeze(0).to(DEVICE)
            model(x)
            feats.append(store["f"].mean(dim=(2, 3)).cpu().numpy()[0])
    h.remove()
    return np.stack(feats)

def shape_texture_dim_ratio(model, layer_name, pairs, n=48):
    ss, st = [], []
    for ip, bin_mask, _ in pairs[:n]:
        img = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_TRAIN, IMG_TRAIN))
        eed = cv2.cvtColor(cv2.Canny(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), 50, 150),
                           cv2.COLOR_GRAY2RGB)
        vor = voronoi_shuffle(img, n_cells=80, seed=SEED + abs(hash(ip)) % 10000)
        ss.append((img, eed)); st.append((img, vor))
    A_s = _layer_feats(model, [a for a, _ in ss], layer_name)
    B_s = _layer_feats(model, [b for _, b in ss], layer_name)
    A_t = _layer_feats(model, [a for a, _ in st], layer_name)
    B_t = _layer_feats(model, [b for _, b in st], layer_name)
    def corr(A, B):
        a = A - A.mean(0, keepdims=True); b = B - B.mean(0, keepdims=True)
        num = (a*b).sum(0); den = np.sqrt((a**2).sum(0)*(b**2).sum(0)) + 1e-8
        return float(np.mean(num/den))
    r_s = corr(A_s, B_s); r_t = corr(A_t, B_t)
    return {"r_shape": r_s, "r_texture": r_t,
            "shape_dim_ratio": r_s / (r_s + r_t + 1e-8)}

# --- 11.4 Grad-CAM [F1] ---
class GradCAM:
    def __init__(self, model, layer):
        self.model, self.layer = model, layer
        self.grad = self.act = None
        layer.register_forward_hook(self._fwd)
        layer.register_full_backward_hook(self._bwd)
    def _fwd(self, m, i, o): self.act = o.detach()
    def _bwd(self, m, gi, go): self.grad = go[0].detach()
    def __call__(self, x):
        self.model.zero_grad()
        d0, *_ = self.model(x); d0.sum().backward()
        w = self.grad.mean(dim=(2, 3), keepdim=True)
        cam = torch.relu((w * self.act).sum(1, keepdim=True))
        cam = torch.nn.functional.interpolate(cam, x.shape[-2:], mode="bilinear",
                                              align_corners=False)
        c = cam[0, 0].cpu().numpy()
        return (c - c.min()) / (c.max() - c.min() + 1e-8)

def gradcam_edge_texture_ratio(model, layer_name, pairs, n=24):
    """[F1] Returns NaN-safe result: skips constant/degenerate CAMs."""
    cam_obj = GradCAM(model, dict(model.named_modules())[layer_name])
    rs_e, rs_t = [], []
    for ip, _, _ in pairs[:n]:
        img = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_TRAIN, IMG_TRAIN))
        eed = cv2.cvtColor(cv2.Canny(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), 50, 150),
                           cv2.COLOR_GRAY2RGB)
        x = norm_tf(transforms.ToTensor()(Image.fromarray(eed))).unsqueeze(0).to(DEVICE)
        try:
            with torch.enable_grad():
                cam = cam_obj(x)
        except Exception:
            continue
        # skip constant/degenerate CAMs
        if not np.isfinite(cam).all() or cam.std() < 1e-8:
            continue
        em = cv2.Canny(cv2.cvtColor(eed, cv2.COLOR_RGB2GRAY), 50, 150).astype(np.float32)
        tm = np.abs(cv2.Laplacian(cv2.cvtColor(eed, cv2.COLOR_RGB2GRAY).astype(np.float32),
                                  cv2.CV_32F))
        if em.std() < 1e-8 or tm.std() < 1e-8:
            continue
        rs_e.append(np.corrcoef(cam.ravel(), em.ravel())[0, 1])
        rs_t.append(np.corrcoef(cam.ravel(), tm.ravel())[0, 1])
    if not rs_e:
        return {"r_edge": float("nan"), "r_texture": float("nan"), "n_valid": 0}
    return {"r_edge": float(np.nanmean(rs_e)),
            "r_texture": float(np.nanmean(rs_t)),
            "n_valid": len(rs_e)}

# --- 11.5 CDRE [F2] — task vs shortcut direction ---
def _fit_linear_probe(X, y, ridge=1e-3):
    X = np.hstack([X, np.ones((X.shape[0], 1))])
    return np.linalg.solve(X.T @ X + ridge*np.eye(X.shape[1]), X.T @ y)

def _collect_features_for_cdre(model, pairs, layer_name, variant, n=96, **variant_kw):
    """Collect layer features and foreground labels under a given input variant."""
    store = {}
    layer = dict(model.named_modules())[layer_name]
    h = layer.register_forward_hook(lambda m, i, o: store.__setitem__("f", o.detach()))
    X, y_fg = [], []
    ds = DiagDS(pairs[:n], variant=variant, **variant_kw)
    loader = DataLoader(ds, batch_size=BATCH, shuffle=False,
                        num_workers=2, pin_memory=True)
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            model(x)
            f = store["f"].mean(dim=(2, 3)).cpu().numpy()
            X.append(f)
            y_fg.append((y[:, 0] > 0.5).float().mean(dim=(1, 2)).numpy())
    h.remove()
    return np.concatenate(X, 0), np.concatenate(y_fg, 0)

def cdre_shortcut_aware(model, pairs, layer_name, shortcut="eed", n=96):
    """
    [F2] CDRE for the shortcut direction = clean-vs-perturbed input.
    Returns |cos(w_task, w_shortcut)| at a given layer.
    """
    Xc, yc = _collect_features_for_cdre(model, pairs, layer_name, "clean", n=n)
    if shortcut == "eed":
        Xp, yp = _collect_features_for_cdre(model, pairs, layer_name, "eed", n=n)
    elif shortcut == "bg_checker":
        Xp, yp = _collect_features_for_cdre(model, pairs, layer_name, "bg_checker",
                                            n=n, tile=16)
    elif shortcut == "bg_noise":
        Xp, yp = _collect_features_for_cdre(model, pairs, layer_name, "bg_noise", n=n)
    else:
        raise ValueError(shortcut)

    X = np.vstack([Xc, Xp])
    y_short = np.concatenate([np.zeros(len(Xc)), np.ones(len(Xp))])
    y_task  = np.concatenate([yc, yp])

    w_task  = _fit_linear_probe(X, y_task)
    w_short = _fit_linear_probe(X, y_short)
    cos = float(np.dot(w_task, w_short) /
                (np.linalg.norm(w_task) * np.linalg.norm(w_short) + 1e-8))
    return abs(cos)

# --- 11.6 Run all probes ---
def encoder_layer_names(model):
    names = dict(model.named_modules())
    return [n for n in ("stage1", "stage4", "stage6") if n in names]

mech_results = {}
for mn in mech_models:
    model = mech_models[mn]
    print(f"\n[mechanism] === {mn} ===")
    e = {}
    try:
        e["cue_conflict_iou"] = cue_conflict_test(model, val_pairs, n=64)
        print(f"  cue-conflict IoU = {e['cue_conflict_iou']:.4f}")
    except Exception as ex: print(f"  cue-conflict failed: {ex}")
    try:
        e["voronoi_ood_iou"] = voronoi_ood_iou(model, val_pairs, n=64)
        print(f"  voronoi OOD IoU = {e['voronoi_ood_iou']:.4f}")
    except Exception as ex: print(f"  voronoi failed: {ex}")

    layers = encoder_layer_names(model)
    e["shape_texture"] = {}
    for ln in layers:
        try:
            r = shape_texture_dim_ratio(model, ln, val_pairs, n=48)
            e["shape_texture"][ln] = r
            print(f"  shape/texture @ {ln}: ratio={r['shape_dim_ratio']:.3f}")
        except Exception as ex: print(f"  shape/texture @ {ln} failed: {ex}")

    e["gradcam"] = {}
    for ln in layers:
        try:
            r = gradcam_edge_texture_ratio(model, ln, val_pairs, n=24)
            e["gradcam"][ln] = r
            re_s = "n/a" if math.isnan(r["r_edge"]) else f"{r['r_edge']:.3f}"
            rt_s = "n/a" if math.isnan(r["r_texture"]) else f"{r['r_texture']:.3f}"
            print(f"  gradcam @ {ln}: r_edge={re_s}  r_tex={rt_s}  (n_valid={r.get('n_valid','?')})")
        except Exception as ex: print(f"  gradcam @ {ln} failed: {ex}")

    try:
        e["cdre_texture"] = {ln: cdre_shortcut_aware(model, val_pairs, ln, "eed", 96)
                             for ln in layers}
        print("  CDRE-texture: " + ", ".join(f"{k}={v:.3f}"
                                              for k, v in e["cdre_texture"].items()))
    except Exception as ex: print(f"  CDRE-texture failed: {ex}")
    try:
        e["cdre_bg"] = {ln: cdre_shortcut_aware(model, val_pairs, ln, "bg_checker", 96)
                        for ln in layers}
        print("  CDRE-bg:      " + ", ".join(f"{k}={v:.3f}"
                                              for k, v in e["cdre_bg"].items()))
    except Exception as ex: print(f"  CDRE-bg failed: {ex}")

    mech_results[mn] = e

with open(os.path.join(MECH_DIR, "mechanism_results.json"), "w") as f:
    json.dump(mech_results, f, indent=2, default=float)

# --- 11.7 Mechanism summary table [F3] ---
print(f"\n{'='*70}\nMECHANISM SUMMARY\n{'='*70}")

def fmt(x):
    """[F3] NaN-safe formatting."""
    if x is None: return "n/a"
    if isinstance(x, float) and (math.isnan(x) or math.isinf(x)): return "n/a"
    if isinstance(x, (int, float)): return f"{x:.3f}"
    return str(x)

rows = []
for mn in mech_results:
    e = mech_results[mn]
    st  = e.get("shape_texture", {}).get("stage6", {})
    gc6 = e.get("gradcam", {}).get("stage6", {})
    cdt = e.get("cdre_texture", {}).get("stage6")
    cdb = e.get("cdre_bg", {}).get("stage6")
    rows.append({
        "model": mn,
        "cue_conflict": fmt(e.get("cue_conflict_iou")),
        "voronoi_ood":  fmt(e.get("voronoi_ood_iou")),
        "shape_ratio@st6": fmt(st.get("shape_dim_ratio")),
        "r_edge@st6":   fmt(gc6.get("r_edge")),
        "r_tex@st6":    fmt(gc6.get("r_texture")),
        "cdre_tex@st6": fmt(cdt),
        "cdre_bg@st6":  fmt(cdb),
    })
print(pd.DataFrame(rows).to_string(index=False))

print("\nHow to read:")
print("  cue_conflict HIGH  -> shape-driven segmentation (good)")
print("  cue_conflict LOW   -> texture causally drives mask (shortcut)")
print("  voronoi_ood LOW    -> shape is necessary; destroying it breaks the model")
print("  shape_ratio@st6 HIGH -> deep representation encodes shape, not texture")
print("  r_edge > r_tex     -> attention follows edges (shape)")
print("  r_tex > r_edge     -> attention follows texture (shortcut)")
print("  cdre_tex@st6 LOW   -> task direction disentangled from EED-shortcut direction")
print("  cdre_bg@st6 LOW    -> task direction disentangled from checker-shortcut direction")

print(f"\nSaved -> {os.path.join(MECH_DIR, 'mechanism_results.json')}")
print("Done.")

Device: cuda
Vanilla ckpt    : /kaggle/input/notebooks/sumn2u/vanilla-u2-net/vanilla_v2/vanilla_v2.pth exists: True
Bg-robust ckpt  : /kaggle/input/notebooks/sumn2u/background-robust/bg_robust_v3/bg_robust_v3.pth exists: True
Fine-tuned dir  : /kaggle/input/notebooks/sumn2u/vanilla-fine-tuned exists: True
Bg-Aug dir      : /kaggle/input/notebooks/sumn2u/vanilla-u-net-with-background-augmentation exists: True
Output dir      : /kaggle/working/eed_test
[discover] candidates under /kaggle/input/notebooks/sumn2u/vanilla-fine-tuned:
     176.44 MB  /kaggle/input/notebooks/sumn2u/vanilla-fine-tuned/vanilla_v2/vanilla_v2.pth
     176.29 MB  /kaggle/input/notebooks/sumn2u/vanilla-fine-tuned/vanilla_v2/u2net_full_weights.pth
Fine-tuned ckpt : /kaggle/input/notebooks/sumn2u/vanilla-fine-tuned/vanilla_v2/vanilla_v2.pth
Fine-tuned arch : /kaggle/input/notebooks/sumn2u/vanilla-fine-tuned/vanilla_v2/u2net_arch.py
[discover] candidates under /kaggle/input/notebooks/sumn2u/vanilla-u-net-with-backgroun

/kaggle/input/notebooks/sumn2u/vanilla-u2-net/vanilla_v2/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')


[vanilla] best threshold: 0.55  clean IoU=0.9685
[vanilla] clean        : 0.9685
[vanilla] eed          : 0.4546
[vanilla] bg_gray      : 0.9878
[vanilla] bg_noise     : 0.4086
[vanilla] bg_gauss25   : 0.9652
[vanilla] bg_blur      : 0.9784
[vanilla] bg_checker   : 0.4079
[vanilla] bg_only      : 0.8787
[vanilla] obj_only     : 0.9878
[vanilla] shift25      : 0.9680
[vanilla] scale0.6     : 0.9598
[vanilla] scale1.5     : 0.9341
[vanilla] crop         : 0.9674
[vanilla] pad          : 0.9579
saved -> /kaggle/working/eed_test/preds_eed_vanilla.png
saved -> /kaggle/working/eed_test/preds_bg_only_vanilla.png
[bg_robust] checkpoint loaded cleanly.

FULL DIAGNOSTIC — BG_ROBUST
[bg_robust] best threshold: 0.45  clean IoU=0.9622
[bg_robust] clean        : 0.9622
[bg_robust] eed          : 0.8905
[bg_robust] bg_gray      : 0.9864
[bg_robust] bg_noise     : 0.9756
[bg_robust] bg_gauss25   : 0.9661
[bg_robust] bg_blur      : 0.9709
[bg_robust] bg_checker   : 0.4062
[bg_robust] bg_only      : 0.8

/kaggle/input/notebooks/sumn2u/vanilla-fine-tuned/vanilla_v2/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')


[finetuned] best threshold: 0.55  clean IoU=0.9685
[finetuned] clean        : 0.9685
[finetuned] eed          : 0.4546
[finetuned] bg_gray      : 0.9878
[finetuned] bg_noise     : 0.4086
[finetuned] bg_gauss25   : 0.9652
[finetuned] bg_blur      : 0.9784
[finetuned] bg_checker   : 0.4079
[finetuned] bg_only      : 0.8787
[finetuned] obj_only     : 0.9878
[finetuned] shift25      : 0.9680
[finetuned] scale0.6     : 0.9598
[finetuned] scale1.5     : 0.9341
[finetuned] crop         : 0.9674
[finetuned] pad          : 0.9579
saved -> /kaggle/working/eed_test/preds_eed_finetuned.png
saved -> /kaggle/working/eed_test/preds_bg_only_finetuned.png
[bg_aug] checkpoint loaded cleanly.

FULL DIAGNOSTIC — BG_AUG


...

  cue-conflict IoU = 0.8853
  voronoi OOD IoU = 0.4499
  shape/texture @ stage1: ratio=0.201
  shape/texture @ stage4: ratio=0.858
  shape/texture @ stage6: ratio=0.878


sys:1: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.


  gradcam @ stage1: r_edge=0.071  r_tex=0.233  (n_valid=24)
  gradcam @ stage4: r_edge=0.239  r_tex=0.358  (n_valid=24)
  gradcam @ stage6: r_edge=0.179  r_tex=0.278  (n_valid=24)
  CDRE-texture: stage1=0.076, stage4=0.051, stage6=0.102
  CDRE-bg:      stage1=0.514, stage4=0.164, stage6=0.241

[mechanism] === bg_robust ===
  cue-conflict IoU = 0.9813
  voronoi OOD IoU = 0.4544
  shape/texture @ stage1: ratio=0.212
  shape/texture @ stage4: ratio=0.802
  shape/texture @ stage6: ratio=0.950
  gradcam @ stage1: r_edge=0.414  r_tex=0.245  (n_valid=23)
  gradcam @ stage4: r_edge=0.216  r_tex=0.355  (n_valid=23)
  gradcam @ stage6: r_edge=0.191  r_tex=0.302  (n_valid=24)
  CDRE-texture: stage1=0.070, stage4=0.090, stage6=0.059
  CDRE-bg:      stage1=0.614, stage4=0.086, stage6=0.193

[mechanism] === bg_aug ===


/kaggle/input/notebooks/sumn2u/vanilla-u-net-with-background-augmentation/vanilla_bgaug/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')


  cue-conflict IoU = 0.8951
  voronoi OOD IoU = 0.4499
  shape/texture @ stage1: ratio=0.180
  shape/texture @ stage4: ratio=0.875
  shape/texture @ stage6: ratio=0.942
  gradcam @ stage1: r_edge=0.302  r_tex=0.183  (n_valid=24)
  gradcam @ stage4: r_edge=0.270  r_tex=0.403  (n_valid=24)
  gradcam @ stage6: r_edge=0.195  r_tex=0.303  (n_valid=24)
  CDRE-texture: stage1=0.030, stage4=0.056, stage6=0.096
  CDRE-bg:      stage1=0.495, stage4=0.169, stage6=0.257

MECHANISM SUMMARY
    model cue_conflict voronoi_ood shape_ratio@st6 r_edge@st6 r_tex@st6 cdre_tex@st6 cdre_bg@st6
  vanilla        0.885       0.450           0.878      0.179     0.278        0.102       0.241
bg_robust        0.981       0.454           0.950      0.191     0.302        0.059       0.193
   bg_aug        0.895       0.450           0.942      0.195     0.303        0.096       0.257

How to read:
  cue_conflict HIGH  -> shape-driven segmentation (good)
  cue_conflict LOW   -> texture causally drives mask (short

In [2]:
# ============================================================
# 11.8 CAUSAL MEDIATION + LINEAR PROBE  [NEW]
# ------------------------------------------------------------
# Two experiments:
#   [E1] Causal tracing: corrupt background, restore stage-k
#        activations from clean run, measure indirect effect.
#   [E2] Linear probe separability: train logistic regression
#        on stage-k features, evaluate under corruption.
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# ---------- helper: single forward with optional activation restore ----------
def _forward_with_restore(model, x, layer_name, restore_act=None, pool=True):
    """Run model(x). If restore_act is given, overwrite layer output with it.
       Returns (logits_or_d0, captured_activation).
    """
    store = {"act": None, "restored": False}
    layer = dict(model.named_modules())[layer_name]

    def hook(m, i, o):
        if restore_act is not None and not store["restored"]:
            store["restored"] = True
            return restore_act
        store["act"] = o.detach()
        return o

    h = layer.register_forward_hook(hook)
    try:
        with torch.no_grad():
            out = model(x)
        d0 = out[0] if isinstance(out, tuple) else out
    finally:
        h.remove()
    return d0, store["act"]


# ---------- [E1] causal tracing ----------
@torch.no_grad()
def causal_tracing(model, pairs, layer_name, corrupt_variant="bg_checker",
                   n=96, thr=0.5, corrupt_kw=None):
    """
    For each val image:
      1. clean run -> cache layer activation, prediction, IoU
      2. corrupt run -> prediction, IoU
      3. restore run -> overwrite layer with clean activation, prediction, IoU
      4. control run -> overwrite layer with activation from a DIFFERENT image
    Returns dict of mean IoU per condition + causal indirect effect.
    """
    corrupt_kw = corrupt_kw or {}
    ds_clean   = DiagDS(pairs[:n], variant="clean", **{})
    ds_corrupt = DiagDS(pairs[:n], variant=corrupt_variant, **corrupt_kw)

    dl_clean   = DataLoader(ds_clean,   batch_size=BATCH, shuffle=False, num_workers=2)
    dl_corrupt = DataLoader(ds_corrupt, batch_size=BATCH, shuffle=False, num_workers=2)

    iou_clean, iou_corrupt, iou_restore, iou_control = [], [], [], []
    all_acts_clean = []

    # Pass 1: collect clean activations (batched) + clean IoU
    for (xc, yc) in dl_clean:
        xc = xc.to(DEVICE)
        d0, act = _forward_with_restore(model, xc, layer_name)
        all_acts_clean.append(act.cpu())
        p = (torch.sigmoid(d0) > thr).float().cpu()
        for k in range(p.shape[0]):
            pk, yk = p[k, 0], yc[k, 0]
            inter = (pk * yk).sum().item()
            union = ((pk + yk) > 0).sum().item()
            iou_clean.append(inter / union if union > 0 else 0.0)

    all_acts_clean = torch.cat(all_acts_clean, dim=0)  # [N, C, H, W]

    # Pass 2: corrupt + restore + control, batched
    idx = 0
    for (xcor, ycor) in dl_corrupt:
        xcor = xcor.to(DEVICE)
        bs = xcor.shape[0]

        # corrupt baseline
        d0_cor, _ = _forward_with_restore(model, xcor, layer_name)
        p_cor = (torch.sigmoid(d0_cor) > thr).float().cpu()

        # restore: use clean activations for the SAME images
        act_restore = all_acts_clean[idx:idx+bs].to(DEVICE)
        d0_res, _ = _forward_with_restore(model, xcor, layer_name,
                                           restore_act=act_restore)
        p_res = (torch.sigmoid(d0_res) > thr).float().cpu()

        # control: use clean activations from a PERMUTED index
        perm = torch.randperm(all_acts_clean.shape[0])[:bs]
        act_ctrl = all_acts_clean[perm].to(DEVICE)
        d0_ctl, _ = _forward_with_restore(model, xcor, layer_name,
                                           restore_act=act_ctrl)
        p_ctl = (torch.sigmoid(d0_ctl) > thr).float().cpu()

        for k in range(bs):
            yk = ycor[k, 0]
            for pred, buf in [(p_cor, iou_corrupt),
                              (p_res, iou_restore),
                              (p_ctl, iou_control)]:
                pk = pred[k, 0]
                inter = (pk * yk).sum().item()
                union = ((pk + yk) > 0).sum().item()
                buf.append(inter / union if union > 0 else 0.0)
        idx += bs

    mc, mcor, mres, mctl = map(float, [np.mean(iou_clean), np.mean(iou_corrupt),
                                        np.mean(iou_restore), np.mean(iou_control)])
    return {
        "clean": mc, "corrupt": mcor, "restore": mres, "control": mctl,
        "IE_restore": mres - mcor,
        "IE_control": mctl - mcor,
        "causal_IE": (mres - mcor) - (mctl - mcor),  # = mres - mctl
        "n": n,
    }


# ---------- [E2] linear probe separability ----------
def _collect_layer_features(model, pairs, layer_name, variant,
                             n=96, thr_dummy=0.5, **variant_kw):
    """Return (features [N,C], foreground_fraction [N], class_idx [N])."""
    ds = DiagDS(pairs[:n], variant=variant, **variant_kw)
    dl = DataLoader(ds, batch_size=BATCH, shuffle=False, num_workers=2)
    layer = dict(model.named_modules())[layer_name]
    store = {"act": None}
    def hook(m, i, o):
        store["act"] = o.detach().mean(dim=(2, 3))  # global avg pool
        return o
    h = layer.register_forward_hook(hook)

    feats, fg, cls_idx = [], [], []
    class_to_idx = {c: i for i, c in enumerate(CLASSES_TRAINING)}
    with torch.no_grad():
        for bi, (x, y) in enumerate(dl):
            x = x.to(DEVICE)
            model(x)
            feats.append(store["act"].cpu().numpy())
            fg.append((y[:, 0] > 0.5).float().mean(dim=(1, 2)).numpy())
            # class label from the pair list (pairs[:n] order preserved)
            for j in range(x.shape[0]):
                global_j = bi * BATCH + j
                if global_j >= n: break
                c = pairs[global_j][2]
                cls_idx.append(class_to_idx.get(c, 0))
    h.remove()
    return (np.concatenate(feats, 0),
            np.concatenate(fg, 0),
            np.array(cls_idx[:len(np.concatenate(feats, 0))]))


def linear_probe_experiment(model, pairs, layer_name,
                             train_pairs_subset, eval_variants=None,
                             n_train=200, n_eval=96):
    """
    Fit logistic regression on CLEAN train features -> class label.
    Evaluate on clean + corrupted features from the SAME model.
    Returns dict of probe accuracy per variant.
    """
    eval_variants = eval_variants or {
        "clean":         ("clean", {}),
        "eed":           ("eed", {}),
        "bg_checker":    ("bg_checker", {"tile": 16}),
        "bg_noise":      ("bg_noise", {}),
        "bg_only":       ("bg_only", {}),
    }

    # --- train probe on clean features ---
    Xtr, _, ytr = _collect_layer_features(model, train_pairs_subset, layer_name,
                                           "clean", n=n_train)
    scaler = StandardScaler().fit(Xtr)
    clf = LogisticRegression(max_iter=2000, C=1.0).fit(scaler.transform(Xtr), ytr)

    out = {}
    for name, (variant, kw) in eval_variants.items():
        Xev, fgev, yev = _collect_layer_features(model, pairs, layer_name,
                                                  variant, n=n_eval, **kw)
        # align yev to Xev length
        m = min(len(Xev), len(yev))
        acc = clf.score(scaler.transform(Xev[:m]), yev[:m])
        # also report mean predicted foreground fraction under this variant
        out[name] = {"probe_acc": float(acc),
                     "mean_fg_frac_gt": float(np.mean(fgev[:m]))}
    # correlate probe drop with CDRE-tex
    drop = out["clean"]["probe_acc"] - out["eed"]["probe_acc"]
    out["_summary"] = {
        "clean_acc": out["clean"]["probe_acc"],
        "eed_acc":   out["eed"]["probe_acc"],
        "eed_drop":  float(drop),
    }
    return out


# ---------- run both experiments ----------
print(f"\n{'='*70}\nCAUSAL MEDIATION + LINEAR PROBE\n{'='*70}")

PROBE_LAYER = "stage6"   # change to "stage1"/"stage4" to probe depth
causal_results = {}
probe_results  = {}

for mn, model in mech_models.items():
    print(f"\n[{mn}] --- causal tracing @ {PROBE_LAYER} ---")
    try:
        ct = causal_tracing(model, val_pairs, PROBE_LAYER,
                            corrupt_variant="bg_checker",
                            n=96, thr=all_results[mn]["best_threshold_clean"]
                                     if all_results.get(mn) else 0.5)
        causal_results[mn] = ct
        print(f"  clean={ct['clean']:.4f}  corrupt={ct['corrupt']:.4f}  "
              f"restore={ct['restore']:.4f}  control={ct['control']:.4f}")
        print(f"  IE_restore={ct['IE_restore']:+.4f}  IE_control={ct['IE_control']:+.4f}  "
              f"causal_IE={ct['causal_IE']:+.4f}")
    except Exception as ex:
        print(f"  causal tracing failed: {ex}"); traceback.print_exc()

    print(f"[{mn}] --- linear probe @ {PROBE_LAYER} ---")
    try:
        # train on a held-out slice of TRAIN pairs (not val) to avoid leakage
        pr = linear_probe_experiment(
            model, val_pairs, PROBE_LAYER,
            train_pairs_subset=train_pairs,
            n_train=200, n_eval=96,
        )
        probe_results[mn] = pr
        s = pr["_summary"]
        print(f"  clean_acc={s['clean_acc']:.3f}  eed_acc={s['eed_acc']:.3f}  "
              f"eed_drop={s['eed_drop']:+.3f}")
        for k, v in pr.items():
            if k.startswith("_"): continue
            print(f"    {k:<12} acc={v['probe_acc']:.3f}  gt_fg={v['mean_fg_frac_gt']:.3f}")
    except Exception as ex:
        print(f"  linear probe failed: {ex}"); traceback.print_exc()

# ---------- save ----------
with open(os.path.join(MECH_DIR, "causal_mediation.json"), "w") as f:
    json.dump(causal_results, f, indent=2, default=float)
with open(os.path.join(MECH_DIR, "linear_probe.json"), "w") as f:
    json.dump(probe_results, f, indent=2, default=float)

# ---------- consolidated table ----------
print(f"\n{'='*70}\nCAUSAL + PROBE SUMMARY (stage6)\n{'='*70}")
rows = []
for mn in mech_models:
    ct = causal_results.get(mn, {})
    pr = probe_results.get(mn, {}).get("_summary", {})
    # pull CDRE-tex from earlier
    cdre = mech_results.get(mn, {}).get("cdre_texture", {}).get("stage6")
    rows.append({
        "model": mn,
        "cdre_tex@st6":  f"{cdre:.3f}" if cdre is not None else "n/a",
        "IE_restore":    f"{ct.get('IE_restore', float('nan')):+.3f}",
        "causal_IE":     f"{ct.get('causal_IE', float('nan')):+.3f}",
        "probe_clean":   f"{pr.get('clean_acc', float('nan')):.3f}",
        "probe_eed":     f"{pr.get('eed_acc', float('nan')):.3f}",
        "probe_drop":    f"{pr.get('eed_drop', float('nan')):+.3f}",
    })
print(pd.DataFrame(rows).to_string(index=False))

print("\nHow to read:")
print("  causal_IE HIGH  -> stage6 causally mediates the shortcut (vanilla/bg_aug expected)")
print("  causal_IE LOW   -> stage6 does NOT carry shortcut (bg_robust expected)")
print("  probe_drop HIGH -> linear readout cannot separate task from shortcut")
print("  probe_drop LOW  -> representation supports invariant readout")
print("  cdre_tex LOW + probe_drop LOW -> orthogonalization (bg_robust)")

# ---------- quick plot ----------
try:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    names = [mn for mn in mech_models]
    x = np.arange(len(names))
    causal_ie = [causal_results.get(mn, {}).get("causal_IE", np.nan) for mn in names]
    probe_drop = [probe_results.get(mn, {}).get("_summary", {}).get("eed_drop", np.nan)
                  for mn in names]
    cdre_tex = [mech_results.get(mn, {}).get("cdre_texture", {}).get("stage6", np.nan)
                for mn in names]

    axes[0].bar(x - 0.2, causal_ie, 0.4, label="causal IE (restore−control)")
    axes[0].bar(x + 0.2, probe_drop, 0.4, label="probe EED drop")
    axes[0].axhline(0, color="k", lw=0.8)
    axes[0].set_xticks(x); axes[0].set_xticklabels(names)
    axes[0].set_title("Causal mediation vs probe drop")
    axes[0].legend(); axes[0].grid(alpha=0.3, axis="y")

    axes[1].scatter(cdre_tex, probe_drop, s=90)
    for i, mn in enumerate(names):
        axes[1].annotate(mn, (cdre_tex[i], probe_drop[i]),
                         textcoords="offset points", xytext=(6, 6))
    axes[1].set_xlabel("CDRE-tex @ stage6")
    axes[1].set_ylabel("linear probe EED drop")
    axes[1].set_title("CDRE predicts readout degradation")
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    out = os.path.join(OUT_DIR, "causal_probe.png")
    plt.savefig(out, dpi=130); plt.close()
    print(f"\nsaved -> {out}")
except Exception as ex:
    print(f"plot failed: {ex}")


CAUSAL MEDIATION + LINEAR PROBE

[vanilla] --- causal tracing @ stage6 ---


/kaggle/input/notebooks/sumn2u/vanilla-u2-net/vanilla_v2/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')


  clean=0.9629  corrupt=0.4481  restore=0.4470  control=0.4446
  IE_restore=-0.0011  IE_control=-0.0035  causal_IE=+0.0025
[vanilla] --- linear probe @ stage6 ---


/kaggle/input/notebooks/sumn2u/vanilla-u2-net/vanilla_v2/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')
/kaggle/input/notebooks/sumn2u/vanilla-u2-net/vanilla_v2/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')
/kaggle/input/notebooks/sumn2u/vanilla-u2-net/vanilla_v2/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')
/kaggle/input/notebooks/sumn2u/vanilla-u2-net/vanilla_v2/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')
/kaggle/input/notebooks/sumn2u/vanilla-u2-net/vanilla_v2/u2net_arch.py:23: UserWarning: 

  clean_acc=0.646  eed_acc=0.219  eed_drop=+0.427
    clean        acc=0.646  gt_fg=0.452
    eed          acc=0.219  gt_fg=0.452
    bg_checker   acc=0.229  gt_fg=0.452
    bg_noise     acc=0.281  gt_fg=0.452
    bg_only      acc=0.260  gt_fg=0.452

[bg_robust] --- causal tracing @ stage6 ---
  clean=0.9594  corrupt=0.4455  restore=0.9481  control=0.4246
  IE_restore=+0.5026  IE_control=-0.0208  causal_IE=+0.5234
[bg_robust] --- linear probe @ stage6 ---
  clean_acc=0.615  eed_acc=0.250  eed_drop=+0.365
    clean        acc=0.615  gt_fg=0.452
    eed          acc=0.250  gt_fg=0.452
    bg_checker   acc=0.229  gt_fg=0.452
    bg_noise     acc=0.406  gt_fg=0.452
    bg_only      acc=0.219  gt_fg=0.452

[bg_aug] --- causal tracing @ stage6 ---


/kaggle/input/notebooks/sumn2u/vanilla-u-net-with-background-augmentation/vanilla_bgaug/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')


  clean=0.9546  corrupt=0.4411  restore=0.4429  control=0.4250
  IE_restore=+0.0018  IE_control=-0.0162  causal_IE=+0.0179
[bg_aug] --- linear probe @ stage6 ---


/kaggle/input/notebooks/sumn2u/vanilla-u-net-with-background-augmentation/vanilla_bgaug/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')
/kaggle/input/notebooks/sumn2u/vanilla-u-net-with-background-augmentation/vanilla_bgaug/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')
/kaggle/input/notebooks/sumn2u/vanilla-u-net-with-background-augmentation/vanilla_bgaug/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')
/kaggle/input/notebooks/sumn2u/vanilla-u-net-with-background-augmentation/vanilla_bgaug/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,

  clean_acc=0.646  eed_acc=0.240  eed_drop=+0.406
    clean        acc=0.646  gt_fg=0.452
    eed          acc=0.240  gt_fg=0.452
    bg_checker   acc=0.240  gt_fg=0.452
    bg_noise     acc=0.438  gt_fg=0.452
    bg_only      acc=0.240  gt_fg=0.452

CAUSAL + PROBE SUMMARY (stage6)
    model cdre_tex@st6 IE_restore causal_IE probe_clean probe_eed probe_drop
  vanilla        0.102     -0.001    +0.002       0.646     0.219     +0.427
bg_robust        0.059     +0.503    +0.523       0.615     0.250     +0.365
   bg_aug        0.096     +0.002    +0.018       0.646     0.240     +0.406

How to read:
  causal_IE HIGH  -> stage6 causally mediates the shortcut (vanilla/bg_aug expected)
  causal_IE LOW   -> stage6 does NOT carry shortcut (bg_robust expected)
  probe_drop HIGH -> linear readout cannot separate task from shortcut
  probe_drop LOW  -> representation supports invariant readout
  cdre_tex LOW + probe_drop LOW -> orthogonalization (bg_robust)

saved -> /kaggle/working/eed_test/cau

In [3]:
# ============================================================
# 11.8 CAUSAL MEDIATION + LINEAR PROBE  (v2 — fixed)
# ------------------------------------------------------------
# Fixes vs v1:
#   [FIX-1] Causal tracing at stage-1 (not stage-6): deep-layer
#           restoration is confounded by upstream corruption.
#   [FIX-2] Two corruption variants: bg_noise (mild, unstructured)
#           and eed (texture removal). bg_checker is retained only
#           as a negative control.
#   [FIX-3] Linear probe target = foreground fraction (fg/bg),
#           matching what U2-Net stage features are trained for.
#   [FIX-4] Restore uses ONLY the same-image clean activation
#           (no index drift) — indices are aligned explicitly.
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# ---------- helper: single forward with optional activation restore ----------
def _forward_with_restore(model, x, layer_name, restore_act=None):
    """Run model(x). If restore_act is given, overwrite layer output with it.
       Returns (d0, captured_activation_or_None)."""
    store = {"act": None, "restored": False}
    layer = dict(model.named_modules())[layer_name]

    def hook(m, i, o):
        if restore_act is not None and not store["restored"]:
            store["restored"] = True
            return restore_act
        store["act"] = o.detach()
        return o

    h = layer.register_forward_hook(hook)
    try:
        with torch.no_grad():
            out = model(x)
        d0 = out[0] if isinstance(out, tuple) else out
    finally:
        h.remove()
    return d0, store["act"]


def _iou_from_pred(p, y, thr=None):
    """p, y are [B,1,H,W] tensors (logits/probs and binary target)."""
    if thr is None:
        p = (torch.sigmoid(p) > 0.5).float()
    else:
        p = (torch.sigmoid(p) > thr).float()
    p = p.cpu(); y = y.cpu()
    out = []
    for k in range(p.shape[0]):
        pk, yk = p[k, 0], y[k, 0]
        inter = (pk * yk).sum().item()
        union = ((pk + yk) > 0).sum().item()
        out.append(inter / union if union > 0 else 0.0)
    return out


# ---------- [E1] causal tracing (v2) ----------
@torch.no_grad()
def causal_tracing(model, pairs, layer_name, corrupt_variant="bg_noise",
                   n=96, thr=0.5, corrupt_kw=None, permute_control=True):
    """
    For each val image:
      1. clean run       -> cache layer activation, IoU
      2. corrupt run     -> IoU
      3. restore run     -> overwrite layer with SAME-image clean activation
      4. control run     -> overwrite layer with PERMUTED clean activation
    Returns mean IoU per condition + causal indirect effect.

    layer_name: use "stage1" for meaningful restoration. stage6 is retained
                only for comparison (restoration is confounded there).
    """
    corrupt_kw = corrupt_kw or {}
    ds_clean   = DiagDS(pairs[:n], variant="clean")
    ds_corrupt = DiagDS(pairs[:n], variant=corrupt_variant, **corrupt_kw)

    dl_clean   = DataLoader(ds_clean,   batch_size=BATCH, shuffle=False, num_workers=2)
    dl_corrupt = DataLoader(ds_corrupt, batch_size=BATCH, shuffle=False, num_workers=2)

    iou_clean, iou_corrupt, iou_restore, iou_control = [], [], [], []
    all_acts_clean = []

    # Pass 1: collect clean activations + clean IoU
    for (xc, yc) in dl_clean:
        xc = xc.to(DEVICE)
        d0, act = _forward_with_restore(model, xc, layer_name)
        all_acts_clean.append(act.cpu())
        iou_clean.extend(_iou_from_pred(d0, yc, thr=thr))

    all_acts_clean = torch.cat(all_acts_clean, dim=0)  # [N, C, H, W]

    # Pass 2: corrupt + restore + control
    idx = 0
    for (xcor, ycor) in dl_corrupt:
        xcor = xcor.to(DEVICE)
        bs = xcor.shape[0]

        # corrupt baseline
        d0_cor, _ = _forward_with_restore(model, xcor, layer_name)
        iou_corrupt.extend(_iou_from_pred(d0_cor, ycor, thr=thr))

        # restore: SAME-image clean activations
        act_restore = all_acts_clean[idx:idx+bs].to(DEVICE)
        d0_res, _ = _forward_with_restore(model, xcor, layer_name,
                                           restore_act=act_restore)
        iou_restore.extend(_iou_from_pred(d0_res, ycor, thr=thr))

        # control: PERMUTED clean activations (same batch, shifted by 1)
        if permute_control:
            perm_local = (torch.arange(bs) + 1) % bs
            act_ctrl = act_restore[perm_local]
        else:
            perm_global = torch.randperm(all_acts_clean.shape[0])[:bs]
            act_ctrl = all_acts_clean[perm_global].to(DEVICE)
        d0_ctl, _ = _forward_with_restore(model, xcor, layer_name,
                                           restore_act=act_ctrl)
        iou_control.extend(_iou_from_pred(d0_ctl, ycor, thr=thr))

        idx += bs

    mc   = float(np.mean(iou_clean))
    mcor = float(np.mean(iou_corrupt))
    mres = float(np.mean(iou_restore))
    mctl = float(np.mean(iou_control))
    return {
        "layer": layer_name, "corrupt_variant": corrupt_variant,
        "clean": mc, "corrupt": mcor, "restore": mres, "control": mctl,
        "IE_restore": mres - mcor,
        "IE_control": mctl - mcor,
        "causal_IE": mres - mctl,        # same-image restore vs permuted control
        "n": n,
    }


# ---------- [E2] linear probe (v2 — fg/bg target) ----------
def _collect_layer_features_v2(model, pairs, layer_name, variant,
                                n=96, **variant_kw):
    """
    Returns:
      feats   : [N, C]  global-avg-pooled layer features
      fg_frac : [N]     ground-truth foreground fraction (from mask)
      cls_idx : [N]     class index (for secondary probe)
    """
    ds = DiagDS(pairs[:n], variant=variant, **variant_kw)
    dl = DataLoader(ds, batch_size=BATCH, shuffle=False, num_workers=2)
    layer = dict(model.named_modules())[layer_name]
    store = {"act": None}
    def hook(m, i, o):
        store["act"] = o.detach().mean(dim=(2, 3))   # global avg pool
        return o
    h = layer.register_forward_hook(hook)

    feats, fg_frac, cls_idx = [], [], []
    class_to_idx = {c: i for i, c in enumerate(CLASSES_TRAINING)}
    with torch.no_grad():
        for bi, (x, y) in enumerate(dl):
            x = x.to(DEVICE)
            model(x)
            feats.append(store["act"].cpu().numpy())
            fg_frac.append((y[:, 0] > 0.5).float().mean(dim=(1, 2)).numpy())
            for j in range(x.shape[0]):
                g = bi * BATCH + j
                if g >= n: break
                cls_idx.append(class_to_idx.get(pairs[g][2], 0))
    h.remove()
    F_ = np.concatenate(feats, 0)
    fg_ = np.concatenate(fg_frac, 0)
    cl_ = np.array(cls_idx[:len(F_)])
    return F_, fg_, cl_


def linear_probe_experiment_v2(model, pairs, layer_name,
                                train_pairs_subset, n_train=200, n_eval=96,
                                eval_variants=None):
    """
    Probe 1: binary fg/bg classification from layer features.
             Target = mask foreground fraction > 0.5 (per-image).
    Probe 2 (secondary): 6-class label from layer features.

    Train on CLEAN features, evaluate on clean + corrupted.
    """
    eval_variants = eval_variants or {
        "clean":      ("clean", {}),
        "eed":        ("eed", {}),
        "bg_noise":   ("bg_noise", {}),
        "bg_checker": ("bg_checker", {"tile": 16}),
        "bg_only":    ("bg_only", {}),
    }

    # --- collect CLEAN train features ---
    Ftr, fgtr, cltr = _collect_layer_features_v2(model, train_pairs_subset,
                                                  layer_name, "clean", n=n_train)
    fgtr_bin = (fgtr > 0.5).astype(int)

    # --- fit fg/bg probe ---
    sc_fg = StandardScaler().fit(Ftr)
    clf_fg = LogisticRegression(max_iter=2000, C=1.0).fit(
        sc_fg.transform(Ftr), fgtr_bin)

    # --- fit class probe (secondary) ---
    sc_cl = StandardScaler().fit(Ftr)
    clf_cl = LogisticRegression(max_iter=2000, C=1.0).fit(
        sc_cl.transform(Ftr), cltr)

    out = {}
    for name, (variant, kw) in eval_variants.items():
        Fev, fgev, clev = _collect_layer_features_v2(model, pairs, layer_name,
                                                      variant, n=n_eval, **kw)
        m = min(len(Fev), len(fgev), len(clev))
        Fev = Fev[:m]; fgev = fgev[:m]; clev = clev[:m]
        fg_bin = (fgev > 0.5).astype(int)

        acc_fg = clf_fg.score(sc_fg.transform(Fev), fg_bin)
        acc_cl = clf_cl.score(sc_cl.transform(Fev), clev)

        out[name] = {
            "probe_fg_acc": float(acc_fg),
            "probe_cls_acc": float(acc_cl),
            "mean_gt_fg_frac": float(np.mean(fgev)),
        }

    out["_summary"] = {
        "clean_fg_acc":  out["clean"]["probe_fg_acc"],
        "eed_fg_acc":    out["eed"]["probe_fg_acc"],
        "noise_fg_acc":  out["bg_noise"]["probe_fg_acc"],
        "checker_fg_acc":out["bg_checker"]["probe_fg_acc"],
        "eed_fg_drop":   out["clean"]["probe_fg_acc"] - out["eed"]["probe_fg_acc"],
        "noise_fg_drop": out["clean"]["probe_fg_acc"] - out["bg_noise"]["probe_fg_acc"],
        "checker_fg_drop": out["clean"]["probe_fg_acc"] - out["bg_checker"]["probe_fg_acc"],
        "clean_cls_acc": out["clean"]["probe_cls_acc"],
        "eed_cls_acc":   out["eed"]["probe_cls_acc"],
    }
    return out


# ---------- run both experiments (v2) ----------
print(f"\n{'='*70}\nCAUSAL MEDIATION + LINEAR PROBE (v2)\n{'='*70}")

# Probe stage-1 for causal tracing (restoration feasible).
# Keep stage-6 as a secondary comparison to document the confound.
CAUSAL_LAYERS = ["stage1", "stage6"]
CORRUPT_VARIANTS = [
    ("bg_noise",   "bg_noise",   {}),                 # mild, unstructured
    ("eed",        "eed",        {}),                 # texture removal
    ("bg_checker", "bg_checker", {"tile": 16}),       # negative control
]
PROBE_LAYER = "stage1"   # where restoration is meaningful

causal_results = {}
probe_results  = {}

for mn, model in mech_models.items():
    thr_m = all_results[mn]["best_threshold_clean"] if all_results.get(mn) else 0.5
    causal_results[mn] = {}

    for layer_name in CAUSAL_LAYERS:
        for cvar, cname, ckw in CORRUPT_VARIANTS:
            tag = f"{layer_name}/{cvar}"
            print(f"\n[{mn}] causal tracing @ {tag} (thr={thr_m:.2f})")
            try:
                ct = causal_tracing(model, val_pairs, layer_name,
                                     corrupt_variant=cvar, n=96,
                                     thr=thr_m, corrupt_kw=ckw)
                causal_results[mn][tag] = ct
                print(f"  clean={ct['clean']:.4f}  corrupt={ct['corrupt']:.4f}  "
                      f"restore={ct['restore']:.4f}  control={ct['control']:.4f}")
                print(f"  IE_restore={ct['IE_restore']:+.4f}  "
                      f"causal_IE={ct['causal_IE']:+.4f}")
            except Exception as ex:
                print(f"  FAILED: {ex}"); traceback.print_exc()

    print(f"\n[{mn}] linear probe @ {PROBE_LAYER}")
    try:
        pr = linear_probe_experiment_v2(
            model, val_pairs, PROBE_LAYER,
            train_pairs_subset=train_pairs,
            n_train=200, n_eval=96,
        )
        probe_results[mn] = pr
        s = pr["_summary"]
        print(f"  FG probe clean={s['clean_fg_acc']:.3f}  "
              f"eed={s['eed_fg_acc']:.3f} (drop {s['eed_fg_drop']:+.3f})  "
              f"noise={s['noise_fg_acc']:.3f} (drop {s['noise_fg_drop']:+.3f})  "
              f"checker={s['checker_fg_acc']:.3f} (drop {s['checker_fg_drop']:+.3f})")
        print(f"  CLS probe clean={s['clean_cls_acc']:.3f}  "
              f"eed={s['eed_cls_acc']:.3f}")
    except Exception as ex:
        print(f"  FAILED: {ex}"); traceback.print_exc()

# ---------- save ----------
with open(os.path.join(MECH_DIR, "causal_mediation_v2.json"), "w") as f:
    json.dump(causal_results, f, indent=2, default=float)
with open(os.path.join(MECH_DIR, "linear_probe_v2.json"), "w") as f:
    json.dump(probe_results, f, indent=2, default=float)

# ---------- consolidated tables ----------
print(f"\n{'='*70}\nCAUSAL TRACING SUMMARY (causal_IE = restore − permuted control)\n{'='*70}")
rows = []
for mn in mech_models:
    for layer_name in CAUSAL_LAYERS:
        row = {"model": mn, "layer": layer_name}
        for cvar, cname, _ in CORRUPT_VARIANTS:
            tag = f"{layer_name}/{cvar}"
            ct = causal_results.get(mn, {}).get(tag, {})
            row[f"{cvar}_IE"] = f"{ct.get('causal_IE', float('nan')):+.3f}"
        rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

print(f"\n{'='*70}\nLINEAR PROBE SUMMARY (FG/bg target, {PROBE_LAYER})\n{'='*70}")
rows = []
for mn in mech_models:
    s = probe_results.get(mn, {}).get("_summary", {})
    cdre = mech_results.get(mn, {}).get("cdre_texture", {}).get(PROBE_LAYER)
    rows.append({
        "model": mn,
        "cdre_tex@"+PROBE_LAYER: f"{cdre:.3f}" if cdre is not None else "n/a",
        "fg_clean":  f"{s.get('clean_fg_acc', float('nan')):.3f}",
        "fg_eed":    f"{s.get('eed_fg_acc', float('nan')):.3f}",
        "fg_noise":  f"{s.get('noise_fg_acc', float('nan')):.3f}",
        "fg_checker":f"{s.get('checker_fg_acc', float('nan')):.3f}",
        "eed_drop":  f"{s.get('eed_fg_drop', float('nan')):+.3f}",
        "noise_drop":f"{s.get('noise_fg_drop', float('nan')):+.3f}",
    })
print(pd.DataFrame(rows).to_string(index=False))

print("\nHow to read (v2):")
print("  causal_IE@stage1 HIGH -> stage1 causally mediates the shortcut")
print("  causal_IE@stage1 LOW  -> shortcut is not read out at stage1")
print("  causal_IE@stage6 HIGH for bg_robust only -> restoration-feasibility confound (documented)")
print("  fg_probe drop LOW     -> linear readout can be invariant (representation supports it)")
print("  fg_probe drop HIGH    -> shortcut contaminates the representation")

# ---------- plots ----------
try:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    names = list(mech_models)
    x = np.arange(len(names))
    w = 0.35

    # left: causal IE at stage1 under bg_noise and eed
    ie_noise = [causal_results.get(mn, {}).get("stage1/bg_noise", {})
                .get("causal_IE", np.nan) for mn in names]
    ie_eed   = [causal_results.get(mn, {}).get("stage1/eed", {})
                .get("causal_IE", np.nan) for mn in names]
    axes[0].bar(x - w/2, ie_noise, w, label="bg_noise")
    axes[0].bar(x + w/2, ie_eed,   w, label="eed")
    axes[0].axhline(0, color="k", lw=0.8)
    axes[0].set_xticks(x); axes[0].set_xticklabels(names)
    axes[0].set_ylabel("causal IE @ stage1")
    axes[0].set_title("Causal mediation at stage1")
    axes[0].legend(); axes[0].grid(alpha=0.3, axis="y")

    # right: fg-probe drop (eed) vs CDRE-tex@stage1
    cdre1 = [mech_results.get(mn, {}).get("cdre_texture", {}).get("stage1", np.nan)
             for mn in names]
    drop1 = [probe_results.get(mn, {}).get("_summary", {}).get("eed_fg_drop", np.nan)
             for mn in names]
    axes[1].scatter(cdre1, drop1, s=110)
    for i, mn in enumerate(names):
        axes[1].annotate(mn, (cdre1[i], drop1[i]),
                         textcoords="offset points", xytext=(6, 6))
    axes[1].set_xlabel("CDRE-tex @ stage1")
    axes[1].set_ylabel("FG probe EED drop")
    axes[1].set_title("CDRE predicts readout degradation")
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    out = os.path.join(OUT_DIR, "causal_probe_v2.png")
    plt.savefig(out, dpi=130); plt.close()
    print(f"\nsaved -> {out}")
except Exception as ex:
    print(f"plot failed: {ex}")

print("\nDone (v2).")


CAUSAL MEDIATION + LINEAR PROBE (v2)

[vanilla] causal tracing @ stage1/bg_noise (thr=0.55)


/kaggle/input/notebooks/sumn2u/vanilla-u2-net/vanilla_v2/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')


  clean=0.9629  corrupt=0.4474  restore=0.9629  control=0.3633
  IE_restore=+0.5155  causal_IE=+0.5995

[vanilla] causal tracing @ stage1/eed (thr=0.55)
  clean=0.9629  corrupt=0.4880  restore=0.9629  control=0.3633
  IE_restore=+0.4748  causal_IE=+0.5995

[vanilla] causal tracing @ stage1/bg_checker (thr=0.55)
  clean=0.9629  corrupt=0.4481  restore=0.9629  control=0.3633
  IE_restore=+0.5148  causal_IE=+0.5995

[vanilla] causal tracing @ stage6/bg_noise (thr=0.55)
  clean=0.9629  corrupt=0.4474  restore=0.4982  control=0.4478
  IE_restore=+0.0508  causal_IE=+0.0504

[vanilla] causal tracing @ stage6/eed (thr=0.55)
  clean=0.9629  corrupt=0.4880  restore=0.8327  control=0.5153
  IE_restore=+0.3447  causal_IE=+0.3174

[vanilla] causal tracing @ stage6/bg_checker (thr=0.55)
  clean=0.9629  corrupt=0.4481  restore=0.4470  control=0.4471
  IE_restore=-0.0011  causal_IE=-0.0000

[vanilla] linear probe @ stage1


/kaggle/input/notebooks/sumn2u/vanilla-u2-net/vanilla_v2/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')
/kaggle/input/notebooks/sumn2u/vanilla-u2-net/vanilla_v2/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')
/kaggle/input/notebooks/sumn2u/vanilla-u2-net/vanilla_v2/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')
/kaggle/input/notebooks/sumn2u/vanilla-u2-net/vanilla_v2/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')
/kaggle/input/notebooks/sumn2u/vanilla-u2-net/vanilla_v2/u2net_arch.py:23: UserWarning: 

  FG probe clean=0.854  eed=0.354 (drop +0.500)  noise=0.354 (drop +0.500)  checker=0.719 (drop +0.135)
  CLS probe clean=0.604  eed=0.229

[bg_robust] causal tracing @ stage1/bg_noise (thr=0.45)
  clean=0.9594  corrupt=0.9740  restore=0.9594  control=0.3621
  IE_restore=-0.0146  causal_IE=+0.5974

[bg_robust] causal tracing @ stage1/eed (thr=0.45)
  clean=0.9594  corrupt=0.8821  restore=0.9594  control=0.3621
  IE_restore=+0.0773  causal_IE=+0.5974

[bg_robust] causal tracing @ stage1/bg_checker (thr=0.45)
  clean=0.9594  corrupt=0.4455  restore=0.9594  control=0.3621
  IE_restore=+0.5139  causal_IE=+0.5974

[bg_robust] causal tracing @ stage6/bg_noise (thr=0.45)
  clean=0.9594  corrupt=0.9740  restore=0.9757  control=0.4799
  IE_restore=+0.0017  causal_IE=+0.4958

[bg_robust] causal tracing @ stage6/eed (thr=0.45)
  clean=0.9594  corrupt=0.8821  restore=0.9464  control=0.4716
  IE_restore=+0.0642  causal_IE=+0.4748

[bg_robust] causal tracing @ stage6/bg_checker (thr=0.45)
  clean=0.

/kaggle/input/notebooks/sumn2u/vanilla-u-net-with-background-augmentation/vanilla_bgaug/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')


  clean=0.9546  corrupt=0.9742  restore=0.9546  control=0.3614
  IE_restore=-0.0196  causal_IE=+0.5932

[bg_aug] causal tracing @ stage1/eed (thr=0.65)
  clean=0.9546  corrupt=0.7484  restore=0.9546  control=0.3614
  IE_restore=+0.2062  causal_IE=+0.5932

[bg_aug] causal tracing @ stage1/bg_checker (thr=0.65)
  clean=0.9546  corrupt=0.4411  restore=0.9546  control=0.3614
  IE_restore=+0.5135  causal_IE=+0.5932

[bg_aug] causal tracing @ stage6/bg_noise (thr=0.65)
  clean=0.9546  corrupt=0.9742  restore=0.9666  control=0.7758
  IE_restore=-0.0076  causal_IE=+0.1908

[bg_aug] causal tracing @ stage6/eed (thr=0.65)
  clean=0.9546  corrupt=0.7484  restore=0.9284  control=0.6480
  IE_restore=+0.1800  causal_IE=+0.2804

[bg_aug] causal tracing @ stage6/bg_checker (thr=0.65)
  clean=0.9546  corrupt=0.4411  restore=0.4429  control=0.4269
  IE_restore=+0.0018  causal_IE=+0.0160

[bg_aug] linear probe @ stage1


/kaggle/input/notebooks/sumn2u/vanilla-u-net-with-background-augmentation/vanilla_bgaug/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')
/kaggle/input/notebooks/sumn2u/vanilla-u-net-with-background-augmentation/vanilla_bgaug/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')
/kaggle/input/notebooks/sumn2u/vanilla-u-net-with-background-augmentation/vanilla_bgaug/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')
/kaggle/input/notebooks/sumn2u/vanilla-u-net-with-background-augmentation/vanilla_bgaug/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,

  FG probe clean=0.854  eed=0.354 (drop +0.500)  noise=0.750 (drop +0.104)  checker=0.323 (drop +0.531)
  CLS probe clean=0.615  eed=0.302

CAUSAL TRACING SUMMARY (causal_IE = restore − permuted control)
    model  layer bg_noise_IE eed_IE bg_checker_IE
  vanilla stage1      +0.600 +0.600        +0.600
  vanilla stage6      +0.050 +0.317        -0.000
bg_robust stage1      +0.597 +0.597        +0.597
bg_robust stage6      +0.496 +0.475        +0.558
   bg_aug stage1      +0.593 +0.593        +0.593
   bg_aug stage6      +0.191 +0.280        +0.016

LINEAR PROBE SUMMARY (FG/bg target, stage1)
    model cdre_tex@stage1 fg_clean fg_eed fg_noise fg_checker eed_drop noise_drop
  vanilla           0.076    0.854  0.354    0.354      0.719   +0.500     +0.500
bg_robust           0.070    0.833  0.646    0.854      0.823   +0.188     -0.021
   bg_aug           0.030    0.854  0.354    0.750      0.323   +0.500     +0.104

How to read (v2):
  causal_IE@stage1 HIGH -> stage1 causally mediates th

In [4]:
# ============================================================
# 11.8 CAUSAL MEDIATION + LINEAR PROBE  (v3 — full depth sweep)
# ------------------------------------------------------------
# v3 changes vs v2:
#   [v3-1] CAUSAL_LAYERS = [stage1, stage4, stage6]
#   [v3-2] Linear probe run at stage1, stage4, stage6
#   [v3-3] Depth-profile table + plot (probe drop vs layer)
#   [v3-4] stage-1 saturation explicitly flagged as non-informative
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# ---------- helper: single forward with optional activation restore ----------
def _forward_with_restore(model, x, layer_name, restore_act=None):
    store = {"act": None, "restored": False}
    layer = dict(model.named_modules())[layer_name]

    def hook(m, i, o):
        if restore_act is not None and not store["restored"]:
            store["restored"] = True
            return restore_act
        store["act"] = o.detach()
        return o

    h = layer.register_forward_hook(hook)
    try:
        with torch.no_grad():
            out = model(x)
        d0 = out[0] if isinstance(out, tuple) else out
    finally:
        h.remove()
    return d0, store["act"]


def _iou_from_pred(p, y, thr=None):
    if thr is None:
        p = (torch.sigmoid(p) > 0.5).float()
    else:
        p = (torch.sigmoid(p) > thr).float()
    p = p.cpu(); y = y.cpu()
    out = []
    for k in range(p.shape[0]):
        pk, yk = p[k, 0], y[k, 0]
        inter = (pk * yk).sum().item()
        union = ((pk + yk) > 0).sum().item()
        out.append(inter / union if union > 0 else 0.0)
    return out


# ---------- [E1] causal tracing (v3) ----------
@torch.no_grad()
def causal_tracing(model, pairs, layer_name, corrupt_variant="bg_noise",
                   n=96, thr=0.5, corrupt_kw=None):
    """
    For each val image:
      1. clean run       -> cache layer activation, IoU
      2. corrupt run     -> IoU
      3. restore run     -> overwrite layer with SAME-image clean activation
      4. control run     -> overwrite layer with PERMUTED clean activation
    Returns mean IoU per condition + causal indirect effect.
    """
    corrupt_kw = corrupt_kw or {}
    ds_clean   = DiagDS(pairs[:n], variant="clean")
    ds_corrupt = DiagDS(pairs[:n], variant=corrupt_variant, **corrupt_kw)

    dl_clean   = DataLoader(ds_clean,   batch_size=BATCH, shuffle=False, num_workers=2)
    dl_corrupt = DataLoader(ds_corrupt, batch_size=BATCH, shuffle=False, num_workers=2)

    iou_clean, iou_corrupt, iou_restore, iou_control = [], [], [], []
    all_acts_clean = []

    for (xc, yc) in dl_clean:
        xc = xc.to(DEVICE)
        d0, act = _forward_with_restore(model, xc, layer_name)
        all_acts_clean.append(act.cpu())
        iou_clean.extend(_iou_from_pred(d0, yc, thr=thr))

    all_acts_clean = torch.cat(all_acts_clean, dim=0)

    idx = 0
    for (xcor, ycor) in dl_corrupt:
        xcor = xcor.to(DEVICE)
        bs = xcor.shape[0]

        d0_cor, _ = _forward_with_restore(model, xcor, layer_name)
        iou_corrupt.extend(_iou_from_pred(d0_cor, ycor, thr=thr))

        act_restore = all_acts_clean[idx:idx+bs].to(DEVICE)
        d0_res, _ = _forward_with_restore(model, xcor, layer_name,
                                           restore_act=act_restore)
        iou_restore.extend(_iou_from_pred(d0_res, ycor, thr=thr))

        perm_local = (torch.arange(bs) + 1) % bs
        act_ctrl = act_restore[perm_local]
        d0_ctl, _ = _forward_with_restore(model, xcor, layer_name,
                                           restore_act=act_ctrl)
        iou_control.extend(_iou_from_pred(d0_ctl, ycor, thr=thr))

        idx += bs

    mc   = float(np.mean(iou_clean))
    mcor = float(np.mean(iou_corrupt))
    mres = float(np.mean(iou_restore))
    mctl = float(np.mean(iou_control))
    return {
        "layer": layer_name, "corrupt_variant": corrupt_variant,
        "clean": mc, "corrupt": mcor, "restore": mres, "control": mctl,
        "IE_restore": mres - mcor,
        "IE_control": mctl - mcor,
        "causal_IE": mres - mctl,
        "n": n,
    }


# ---------- [E2] linear probe (v3 — depth sweep) ----------
def _collect_layer_features_v2(model, pairs, layer_name, variant,
                                n=96, **variant_kw):
    ds = DiagDS(pairs[:n], variant=variant, **variant_kw)
    dl = DataLoader(ds, batch_size=BATCH, shuffle=False, num_workers=2)
    layer = dict(model.named_modules())[layer_name]
    store = {"act": None}
    def hook(m, i, o):
        store["act"] = o.detach().mean(dim=(2, 3))
        return o
    h = layer.register_forward_hook(hook)

    feats, fg_frac, cls_idx = [], [], []
    class_to_idx = {c: i for i, c in enumerate(CLASSES_TRAINING)}
    with torch.no_grad():
        for bi, (x, y) in enumerate(dl):
            x = x.to(DEVICE)
            model(x)
            feats.append(store["act"].cpu().numpy())
            fg_frac.append((y[:, 0] > 0.5).float().mean(dim=(1, 2)).numpy())
            for j in range(x.shape[0]):
                g = bi * BATCH + j
                if g >= n: break
                cls_idx.append(class_to_idx.get(pairs[g][2], 0))
    h.remove()
    F_ = np.concatenate(feats, 0)
    fg_ = np.concatenate(fg_frac, 0)
    cl_ = np.array(cls_idx[:len(F_)])
    return F_, fg_, cl_


def linear_probe_at_layer(model, pairs, layer_name, train_pairs_subset,
                           n_train=200, n_eval=96, eval_variants=None):
    eval_variants = eval_variants or {
        "clean":      ("clean", {}),
        "eed":        ("eed", {}),
        "bg_noise":   ("bg_noise", {}),
        "bg_checker": ("bg_checker", {"tile": 16}),
        "bg_only":    ("bg_only", {}),
    }

    Ftr, fgtr, cltr = _collect_layer_features_v2(model, train_pairs_subset,
                                                  layer_name, "clean", n=n_train)
    fgtr_bin = (fgtr > 0.5).astype(int)

    sc_fg = StandardScaler().fit(Ftr)
    clf_fg = LogisticRegression(max_iter=2000, C=1.0).fit(
        sc_fg.transform(Ftr), fgtr_bin)

    sc_cl = StandardScaler().fit(Ftr)
    clf_cl = LogisticRegression(max_iter=2000, C=1.0).fit(
        sc_cl.transform(Ftr), cltr)

    out = {"layer": layer_name}
    for name, (variant, kw) in eval_variants.items():
        Fev, fgev, clev = _collect_layer_features_v2(model, pairs, layer_name,
                                                      variant, n=n_eval, **kw)
        m = min(len(Fev), len(fgev), len(clev))
        Fev = Fev[:m]; fgev = fgev[:m]; clev = clev[:m]
        fg_bin = (fgev > 0.5).astype(int)

        out[name] = {
            "probe_fg_acc": float(clf_fg.score(sc_fg.transform(Fev), fg_bin)),
            "probe_cls_acc": float(clf_cl.score(sc_cl.transform(Fev), clev)),
            "mean_gt_fg_frac": float(np.mean(fgev)),
        }

    out["_summary"] = {
        "clean_fg_acc":  out["clean"]["probe_fg_acc"],
        "eed_fg_acc":    out["eed"]["probe_fg_acc"],
        "noise_fg_acc":  out["bg_noise"]["probe_fg_acc"],
        "checker_fg_acc":out["bg_checker"]["probe_fg_acc"],
        "eed_fg_drop":   out["clean"]["probe_fg_acc"] - out["eed"]["probe_fg_acc"],
        "noise_fg_drop": out["clean"]["probe_fg_acc"] - out["bg_noise"]["probe_fg_acc"],
        "checker_fg_drop": out["clean"]["probe_fg_acc"] - out["bg_checker"]["probe_fg_acc"],
        "clean_cls_acc": out["clean"]["probe_cls_acc"],
        "eed_cls_acc":   out["eed"]["probe_cls_acc"],
    }
    return out


# ---------- run (v3) ----------
print(f"\n{'='*70}\nCAUSAL MEDIATION + LINEAR PROBE (v3 — full depth sweep)\n{'='*70}")

CAUSAL_LAYERS   = ["stage1", "stage4", "stage6"]
PROBE_LAYERS    = ["stage1", "stage4", "stage6"]
CORRUPT_VARIANTS = [
    ("bg_noise",   "bg_noise",   {}),
    ("eed",        "eed",        {}),
    ("bg_checker", "bg_checker", {"tile": 16}),
]

causal_results = {}   # causal_results[model][layer/variant]
probe_results  = {}   # probe_results[model][layer]

for mn, model in mech_models.items():
    thr_m = all_results[mn]["best_threshold_clean"] if all_results.get(mn) else 0.5
    causal_results[mn] = {}
    probe_results[mn]  = {}

    # --- causal tracing across layers x variants ---
    for layer_name in CAUSAL_LAYERS:
        for cvar, cname, ckw in CORRUPT_VARIANTS:
            tag = f"{layer_name}/{cvar}"
            print(f"\n[{mn}] causal tracing @ {tag} (thr={thr_m:.2f})")
            try:
                ct = causal_tracing(model, val_pairs, layer_name,
                                     corrupt_variant=cvar, n=96,
                                     thr=thr_m, corrupt_kw=ckw)
                causal_results[mn][tag] = ct
                print(f"  clean={ct['clean']:.4f}  corrupt={ct['corrupt']:.4f}  "
                      f"restore={ct['restore']:.4f}  control={ct['control']:.4f}")
                print(f"  IE_restore={ct['IE_restore']:+.4f}  "
                      f"causal_IE={ct['causal_IE']:+.4f}")
            except Exception as ex:
                print(f"  FAILED: {ex}"); traceback.print_exc()

    # --- linear probe across layers ---
    for layer_name in PROBE_LAYERS:
        print(f"\n[{mn}] linear probe @ {layer_name}")
        try:
            pr = linear_probe_at_layer(model, val_pairs, layer_name,
                                        train_pairs_subset=train_pairs,
                                        n_train=200, n_eval=96)
            probe_results[mn][layer_name] = pr
            s = pr["_summary"]
            print(f"  FG  clean={s['clean_fg_acc']:.3f}  "
                  f"eed={s['eed_fg_acc']:.3f} (drop {s['eed_fg_drop']:+.3f})  "
                  f"noise={s['noise_fg_acc']:.3f} (drop {s['noise_fg_drop']:+.3f})  "
                  f"checker={s['checker_fg_acc']:.3f} (drop {s['checker_fg_drop']:+.3f})")
        except Exception as ex:
            print(f"  FAILED: {ex}"); traceback.print_exc()

# ---------- save ----------
with open(os.path.join(MECH_DIR, "causal_mediation_v3.json"), "w") as f:
    json.dump(causal_results, f, indent=2, default=float)
with open(os.path.join(MECH_DIR, "linear_probe_v3.json"), "w") as f:
    json.dump(probe_results, f, indent=2, default=float)

# ---------- consolidated causal table ----------
print(f"\n{'='*70}\nCAUSAL TRACING SUMMARY v3 (causal_IE = restore − permuted control)\n{'='*70}")
rows = []
for mn in mech_models:
    for layer_name in CAUSAL_LAYERS:
        row = {"model": mn, "layer": layer_name}
        for cvar, _, _ in CORRUPT_VARIANTS:
            ct = causal_results.get(mn, {}).get(f"{layer_name}/{cvar}", {})
            row[cvar] = f"{ct.get('causal_IE', float('nan')):+.3f}"
        rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

print("\nNote: stage1 causal_IE saturates near +0.60 for all models — "
      "restoration is trivially complete at this depth. "
      "Discriminative signal is at stage4/stage6.")

# ---------- consolidated probe table ----------
print(f"\n{'='*70}\nLINEAR PROBE SUMMARY v3 (FG/bg target, depth sweep)\n{'='*70}")
rows = []
for mn in mech_models:
    for layer_name in PROBE_LAYERS:
        pr = probe_results.get(mn, {}).get(layer_name, {})
        s = pr.get("_summary", {})
        cdre = mech_results.get(mn, {}).get("cdre_texture", {}).get(layer_name)
        rows.append({
            "model": mn, "layer": layer_name,
            "cdre_tex": f"{cdre:.3f}" if cdre is not None else "n/a",
            "fg_clean":  f"{s.get('clean_fg_acc', float('nan')):.3f}",
            "fg_eed":    f"{s.get('eed_fg_acc', float('nan')):.3f}",
            "fg_noise":  f"{s.get('noise_fg_acc', float('nan')):.3f}",
            "fg_checker":f"{s.get('checker_fg_acc', float('nan')):.3f}",
            "eed_drop":  f"{s.get('eed_fg_drop', float('nan')):+.3f}",
            "noise_drop":f"{s.get('noise_fg_drop', float('nan')):+.3f}",
            "checker_drop": f"{s.get('checker_fg_drop', float('nan')):+.3f}",
        })
print(pd.DataFrame(rows).to_string(index=False))

# ---------- depth-profile table ----------
print(f"\n{'='*70}\nDEPTH PROFILE — FG probe drop vs layer\n{'='*70}")
rows = []
for mn in mech_models:
    for layer_name in PROBE_LAYERS:
        s = probe_results.get(mn, {}).get(layer_name, {}).get("_summary", {})
        rows.append({
            "model": mn, "layer": layer_name,
            "eed_drop":   f"{s.get('eed_fg_drop', float('nan')):+.3f}",
            "noise_drop": f"{s.get('noise_fg_drop', float('nan')):+.3f}",
        })
print(pd.DataFrame(rows).to_string(index=False))

# ---------- plots ----------
try:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
    names = list(mech_models)
    x = np.arange(len(names))
    w = 0.25

    # (a) causal IE @ stage4 under all corruptions
    for i, (cvar, _, _) in enumerate(CORRUPT_VARIANTS):
        vals = [causal_results.get(mn, {}).get(f"stage4/{cvar}", {})
                .get("causal_IE", np.nan) for mn in names]
        axes[0].bar(x + (i-1)*w, vals, w, label=cvar)
    axes[0].axhline(0, color="k", lw=0.8)
    axes[0].set_xticks(x); axes[0].set_xticklabels(names)
    axes[0].set_ylabel("causal IE @ stage4")
    axes[0].set_title("(a) Causal mediation @ stage4")
    axes[0].legend(); axes[0].grid(alpha=0.3, axis="y")

    # (b) depth profile — FG probe EED drop
    for mn in names:
        drops = [probe_results.get(mn, {}).get(l, {}).get("_summary", {})
                 .get("eed_fg_drop", np.nan) for l in PROBE_LAYERS]
        axes[1].plot(PROBE_LAYERS, drops, marker="o", label=mn)
    axes[1].set_ylabel("FG probe EED drop")
    axes[1].set_title("(b) Readout fragility vs depth (EED)")
    axes[1].legend(); axes[1].grid(alpha=0.3)

    # (c) depth profile — FG probe noise drop
    for mn in names:
        drops = [probe_results.get(mn, {}).get(l, {}).get("_summary", {})
                 .get("noise_fg_drop", np.nan) for l in PROBE_LAYERS]
        axes[2].plot(PROBE_LAYERS, drops, marker="o", label=mn)
    axes[2].axhline(0, color="k", lw=0.8)
    axes[2].set_ylabel("FG probe noise drop")
    axes[2].set_title("(c) Readout fragility vs depth (noise)")
    axes[2].legend(); axes[2].grid(alpha=0.3)

    plt.tight_layout()
    out = os.path.join(OUT_DIR, "causal_probe_v3.png")
    plt.savefig(out, dpi=130); plt.close()
    print(f"\nsaved -> {out}")
except Exception as ex:
    print(f"plot failed: {ex}")

print("\nDone (v3).")


CAUSAL MEDIATION + LINEAR PROBE (v3 — full depth sweep)

[vanilla] causal tracing @ stage1/bg_noise (thr=0.55)


/kaggle/input/notebooks/sumn2u/vanilla-u2-net/vanilla_v2/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')


  clean=0.9629  corrupt=0.4474  restore=0.9629  control=0.3633
  IE_restore=+0.5155  causal_IE=+0.5995

[vanilla] causal tracing @ stage1/eed (thr=0.55)
  clean=0.9629  corrupt=0.4880  restore=0.9629  control=0.3633
  IE_restore=+0.4748  causal_IE=+0.5995

[vanilla] causal tracing @ stage1/bg_checker (thr=0.55)
  clean=0.9629  corrupt=0.4481  restore=0.9629  control=0.3633
  IE_restore=+0.5148  causal_IE=+0.5995

[vanilla] causal tracing @ stage4/bg_noise (thr=0.55)
  clean=0.9629  corrupt=0.4474  restore=0.8929  control=0.3558
  IE_restore=+0.4455  causal_IE=+0.5371

[vanilla] causal tracing @ stage4/eed (thr=0.55)
  clean=0.9629  corrupt=0.4880  restore=0.9499  control=0.4995
  IE_restore=+0.4619  causal_IE=+0.4504

[vanilla] causal tracing @ stage4/bg_checker (thr=0.55)
  clean=0.9629  corrupt=0.4481  restore=0.7112  control=0.3203
  IE_restore=+0.2631  causal_IE=+0.3909

[vanilla] causal tracing @ stage6/bg_noise (thr=0.55)
  clean=0.9629  corrupt=0.4474  restore=0.4982  control=0.

/kaggle/input/notebooks/sumn2u/vanilla-u2-net/vanilla_v2/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')
/kaggle/input/notebooks/sumn2u/vanilla-u2-net/vanilla_v2/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')
/kaggle/input/notebooks/sumn2u/vanilla-u2-net/vanilla_v2/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')
/kaggle/input/notebooks/sumn2u/vanilla-u2-net/vanilla_v2/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')
/kaggle/input/notebooks/sumn2u/vanilla-u2-net/vanilla_v2/u2net_arch.py:23: UserWarning: 

  FG  clean=0.854  eed=0.354 (drop +0.500)  noise=0.354 (drop +0.500)  checker=0.719 (drop +0.135)

[vanilla] linear probe @ stage4


...

  FG  clean=0.854  eed=0.354 (drop +0.500)  noise=0.750 (drop +0.104)  checker=0.323 (drop +0.531)

[bg_aug] linear probe @ stage4


/kaggle/input/notebooks/sumn2u/vanilla-u-net-with-background-augmentation/vanilla_bgaug/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')
/kaggle/input/notebooks/sumn2u/vanilla-u-net-with-background-augmentation/vanilla_bgaug/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')
/kaggle/input/notebooks/sumn2u/vanilla-u-net-with-background-augmentation/vanilla_bgaug/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')
/kaggle/input/notebooks/sumn2u/vanilla-u-net-with-background-augmentation/vanilla_bgaug/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,

  FG  clean=0.896  eed=0.760 (drop +0.135)  noise=0.615 (drop +0.281)  checker=0.354 (drop +0.542)

[bg_aug] linear probe @ stage6


/kaggle/input/notebooks/sumn2u/vanilla-u-net-with-background-augmentation/vanilla_bgaug/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')
/kaggle/input/notebooks/sumn2u/vanilla-u-net-with-background-augmentation/vanilla_bgaug/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')
/kaggle/input/notebooks/sumn2u/vanilla-u-net-with-background-augmentation/vanilla_bgaug/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,size=tar.shape[2:],mode='bilinear')
/kaggle/input/notebooks/sumn2u/vanilla-u-net-with-background-augmentation/vanilla_bgaug/u2net_arch.py:23: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  src = F.upsample(src,

  FG  clean=0.927  eed=0.562 (drop +0.365)  noise=0.927 (drop +0.000)  checker=0.354 (drop +0.573)

CAUSAL TRACING SUMMARY v3 (causal_IE = restore − permuted control)
    model  layer bg_noise    eed bg_checker
  vanilla stage1   +0.600 +0.600     +0.600
  vanilla stage4   +0.537 +0.450     +0.391
  vanilla stage6   +0.050 +0.317     -0.000
bg_robust stage1   +0.597 +0.597     +0.597
bg_robust stage4   +0.599 +0.586     +0.604
bg_robust stage6   +0.496 +0.475     +0.558
   bg_aug stage1   +0.593 +0.593     +0.593
   bg_aug stage4   +0.595 +0.568     +0.581
   bg_aug stage6   +0.191 +0.280     +0.016

Note: stage1 causal_IE saturates near +0.60 for all models — restoration is trivially complete at this depth. Discriminative signal is at stage4/stage6.

LINEAR PROBE SUMMARY v3 (FG/bg target, depth sweep)
    model  layer cdre_tex fg_clean fg_eed fg_noise fg_checker eed_drop noise_drop checker_drop
  vanilla stage1    0.076    0.854  0.354    0.354      0.719   +0.500     +0.500       +0.